# CMIP6 extraction, first version (superseded)

**Alwar groundwater study · 14-block domain (LGD 574–587)**

First extraction of the Mishra et al. CMIP6 data for the 14 blocks (SSP2-4.5 only), with the exploration of other CMIP6 sources (INDRA-CMIP6, NEX-GDDP). The block weights and the 14-block selection made here are still used; the extracted daily values are not.

> ⚠️ **Superseded — kept as a record of the investigation.** These cells read grid columns at `3 + column_index_zero_based`, which are about 0.75° north of Alwar (the index already counts the three date columns). The corrected data are produced by `02_climate/03_climate_forcing_corrected.ipynb`. Do not use outputs of this notebook as model input.

| | |
|---|---|
| **Reads** | `gw_framework/climate_forcing/Vimal_Mishra_CMIP6/India.tar.xz`, `gw_framework/Shapefile/LGD_Blocks.geojson` |
| **Writes** | `Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/` (`Area_weighted_spatial_selection/`, `Alwar_block_daily_forcing/`, `Climate_Indicators/`) |
| **Cells** | 60 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Other CMIP6 sources explored (not used downstream)
- 2. Download and extract the Mishra CMIP6 archive
- 3. Grid cells and block weights
- 4. Block-weighted daily forcing
- 5. Quality control and Tmax < Tmin diagnosis
- 6. Rebuild from source (unfinished; superseded)

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["xarray", "netCDF4", "geopandas"], modflow=False)
del TOKEN, URL


## 1. Other CMIP6 sources explored (not used downstream)

**[orig #43]** List the INDRA-CMIP6 folder  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/INDRA-CMIP6"
)

print("Root exists:", ROOT.exists())
print("\nFolder structure:\n")

for p in sorted(ROOT.rglob("*")):
    relative = p.relative_to(ROOT)
    if p.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        print(f"[FILE] {relative}  |  {p.stat().st_size / 1e6:.2f} MB")

**[orig #45]** INDRA-CMIP6 file types and climate-relevant files  
<sub>✅ ran OK</sub>

In [ ]:
from collections import Counter

files = [p for p in ROOT.rglob("*") if p.is_file()]

print("Total files:", len(files))
print("\nFile extensions:")
print(Counter(p.suffix.lower() for p in files))

keywords = [
    "pr", "precip", "rain", "tasmax", "tasmin",
    "tmax", "tmin", "ssp245", "ssp585", "historical"
]

print("\nFiles relevant to climate analysis:")
for p in files:
    if any(k in str(p).lower() for k in keywords):
        print(p.relative_to(ROOT))

**[orig #46]** Open INDRA-CMIP6 NetCDF files; dimensions  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import xarray as xr

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/INDRA-CMIP6"
)

files_to_check = [
    ROOT / "pr/ensemble/pr_ens_historical.nc",
    ROOT / "pr/ensemble/pr_ens_ssp245.nc",
    ROOT / "tasmax/ensemble/tasmax_ens_historical.nc",
    ROOT / "tasmax/ensemble/tasmax_ens_ssp245.nc",
    ROOT / "tasmin/ensemble/tasmin_ens_historical.nc",
    ROOT / "tasmin/ensemble/tasmin_ens_ssp245.nc",
    ROOT / "sample_data/sample_yearly_pr_historical_ACCESS-CM2.nc",
]

for path in files_to_check:
    print("\n" + "=" * 75)
    print("FILE:", path.relative_to(ROOT))

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        with xr.open_dataset(path) as ds:
            print("Dimensions:", dict(ds.sizes))
            print("Variables:", list(ds.data_vars))

            for name, var in ds.data_vars.items():
                print(
                    f"Variable: {name}",
                    "| units:", var.attrs.get("units"),
                    "| long_name:", var.attrs.get("long_name")
                )

            for coord in ds.coords:
                c = ds[coord]
                if c.ndim == 1 and c.size > 1:
                    print(
                        f"Coordinate {coord}:",
                        c.values[0], "to", c.values[-1],
                        "| size:", c.size
                    )
                elif c.ndim == 0:
                    print(f"Coordinate {coord}:", c.values)

            print("Global attributes:", {
                k: ds.attrs[k]
                for k in list(ds.attrs)[:10]
            })

    except Exception as e:
        print("Could not inspect:", repr(e))

**[orig #47]** INDRA-CMIP6 values over a Rajasthan box  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import xarray as xr
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/INDRA-CMIP6"
)

# Approximate Rajasthan bounding box for an initial check.
# This is NOT the final state boundary.
LAT_MIN, LAT_MAX = 23.0, 30.5
LON_MIN, LON_MAX = 69.0, 78.5

files = {
    "pr_hist": ROOT / "pr/ensemble/pr_ens_historical.nc",
    "pr_ssp245": ROOT / "pr/ensemble/pr_ens_ssp245.nc",
    "tasmax_hist": ROOT / "tasmax/ensemble/tasmax_ens_historical.nc",
    "tasmax_ssp245": ROOT / "tasmax/ensemble/tasmax_ens_ssp245.nc",
    "tasmin_hist": ROOT / "tasmin/ensemble/tasmin_ens_historical.nc",
    "tasmin_ssp245": ROOT / "tasmin/ensemble/tasmin_ens_ssp245.nc",
}

for label, path in files.items():
    print("\n" + "=" * 65)
    print(label, "|", path.name)

    with xr.open_dataset(path) as ds:
        varname = next(iter(ds.data_vars))
        da = ds[varname]

        print("Variable:", varname)
        print("Attributes:", da.attrs)
        print("Encoding units:", da.encoding.get("units"))
        print("Time units:", ds["time"].attrs.get("units"))
        print("Calendar:", ds["time"].encoding.get("calendar"))

        # Latitude in these files is descending.
        subset = da.sel(
            lat=slice(LAT_MAX, LAT_MIN),
            lon=slice(LON_MIN, LON_MAX)
        )

        # Use only a few dates for a quick data-value check.
        sample = subset.isel(time=slice(0, min(10, subset.sizes["time"])))
        values = sample.values
        finite = values[np.isfinite(values)]

        if finite.size:
            print("First 10 time steps, spatial subset:")
            print("Min:", float(finite.min()))
            print("Max:", float(finite.max()))
            print("Mean:", float(finite.mean()))
        else:
            print("No finite values in this sample.")

        print("Subset dimensions:", dict(subset.sizes))

**[orig #48]** Check INDRA-CMIP6 historical / SSP245 files for ACCESS-CM2  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import xarray as xr

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/INDRA-CMIP6"
)

check_files = [
    ROOT / "pr/ensemble/pr_ens_historical.nc",
    ROOT / "pr/ACCESS-CM2/historical_DBCCA.nc",
    ROOT / "pr/ACCESS-CM2/ssp245_DBCCA.nc",
    ROOT / "tasmax/ACCESS-CM2/historical_DBCCA.nc",
    ROOT / "tasmin/ACCESS-CM2/historical_DBCCA.nc",
    ROOT / "sample_data/sample_yearly_pr_historical_ACCESS-CM2.nc",
]

for path in check_files:
    print("\n" + "=" * 70)
    print("FILE:", path.relative_to(ROOT))

    if not path.exists():
        print("File not found")
        continue

    with xr.open_dataset(path) as ds:
        print("Global attributes:", ds.attrs)
        print("Dimensions:", dict(ds.sizes))

        for name, da in ds.data_vars.items():
            print(f"\nVariable: {name}")
            print("Attributes:", da.attrs)
            print("Encoding:", {
                k: da.encoding.get(k)
                for k in ["dtype", "units", "scale_factor",
                          "add_offset", "_FillValue"]
            })

        if "time" in ds.coords:
            print("Time encoding:", ds["time"].encoding)
            print("Time attributes:", ds["time"].attrs)

**[orig #146]** Step 19A — climate–groundwater compatibility audit  
<sub>✅ ran OK</sub>

In [ ]:

# ============================================================
# STEP 19A — CLIMATE–GROUNDWATER DATA COMPATIBILITY AUDIT
# Study area: Alwar, Rajasthan (14 selected blocks)
# Purpose: File, date, variable, unit, and spatial compatibility
# Safety: Reads source files only; writes audit outputs separately
# ============================================================

from pathlib import Path
import os
import re
import json
import warnings
import numpy as np
import pandas as pd

# ---------------------------
# 1. Mount Google Drive
# ---------------------------
from google.colab import drive
drive.mount("/content/drive")

# ---------------------------
# 2. Define project paths
# ---------------------------
ROOT = Path("/content/drive/MyDrive/gw_framework")

CLIMATE_DIR = ROOT / "data" / "Cmip6 indices" / "CMIP6 Indices"
COMPARISON_DIR = CLIMATE_DIR / "Comparison" / "ssp245"
HISTORICAL_DIR = CLIMATE_DIR / "Interpolation_20112023"

FORCING_DIR = ROOT / "climate_forcing" / "csv"
GROUNDWATER_DIR = ROOT / "data" / "Rajasthan_gw_data"

# Existing spatially restricted groundwater observations
GW_CANDIDATES = [
    ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery/"
         "Step_15_Spatial_Trend_Restriction/"
         "02_spatially_restricted_groundwater_observations.csv",
    GROUNDWATER_DIR / "QC_outputs/groundwater_long_seasonal.csv",
    GROUNDWATER_DIR /
        "Groundwater_Cleaning_According_to_User_Rules/"
        "groundwater_cleaned_long_seasonal.csv",
]

# Known climate files from the project inventory
CLIMATE_CANDIDATES = [
    COMPARISON_DIR / "Tmax_Hybrid_SSP245_02112023.csv",
    COMPARISON_DIR / "Tmin_Hybrid_SSP245_02112023.csv",
    COMPARISON_DIR / "PCP_Hybrid_SSP245_02112023.csv",
    FORCING_DIR / "ACCESS-CM2_ssp245_bc_daily.csv",
    FORCING_DIR / "ACCESS-CM2_ssp585_bc_daily.csv",
    HISTORICAL_DIR / "Tmax_hist_interpolation.csv",
    HISTORICAL_DIR / "Tmin_hist_interpolation.csv",
    HISTORICAL_DIR / "pcp_ssp245_interpolation.csv",
]

OUT = (
    ROOT / "data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_19A_Climate_Groundwater_Audit"
)
OUT.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT)
print("Original datasets will not be edited.")

# ---------------------------
# 3. Robust file reader
# ---------------------------
def read_table(path, nrows=None):
    """Try common CSV separators; support Excel and NetCDF inventory."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, nrows=nrows)

    if suffix in [".csv", ".txt"]:
        # sep=None uses the Python CSV delimiter sniffer.
        # Try common separators if automatic detection fails.
        for sep in [None, ",", "|", ";", "\t"]:
            try:
                df = pd.read_csv(
                    path, sep=sep, engine="python",
                    nrows=nrows, on_bad_lines="warn"
                )
                if len(df.columns) > 1:
                    return df
            except Exception:
                pass

        # Last attempt: preserve a one-column result for diagnosis
        return pd.read_csv(path, nrows=nrows)

    raise ValueError(f"Unsupported table format: {suffix}")


def find_col(df, candidates):
    """Find a column using case-insensitive exact/partial matches."""
    cols = list(df.columns)
    normalized = {
        str(c).strip().lower().replace(" ", "_"): c for c in cols
    }

    for candidate in candidates:
        key = candidate.lower().replace(" ", "_")
        if key in normalized:
            return normalized[key]

    for c in cols:
        name = str(c).strip().lower().replace(" ", "_")
        for candidate in candidates:
            key = candidate.lower().replace(" ", "_")
            if key in name:
                return c

    return None


def parse_dates(series):
    """Parse common ISO and day-first date formats."""
    s = series.astype(str).str.strip()
    parsed = pd.to_datetime(s, errors="coerce", format="mixed")
    # If parsing failed for many values, test day-first dates
    if parsed.notna().sum() < max(1, int(0.5 * len(s))):
        parsed2 = pd.to_datetime(
            s, errors="coerce", dayfirst=True, format="mixed"
        )
        if parsed2.notna().sum() > parsed.notna().sum():
            parsed = parsed2
    return parsed


# ---------------------------
# 4. Check expected file paths
# ---------------------------
inventory = []

for p in CLIMATE_CANDIDATES + GW_CANDIDATES:
    inventory.append({
        "file": str(p),
        "exists": p.exists(),
        "size_MB": round(p.stat().st_size / (1024**2), 3)
                    if p.exists() else np.nan,
        "suffix": p.suffix.lower(),
        "role": "groundwater" if p in GW_CANDIDATES else "climate"
    })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(OUT / "01_file_inventory.csv", index=False)

print("\nFILE INVENTORY")
display(inventory_df)

# Choose the first existing groundwater dataset in the list.
GW_PATH = next((p for p in GW_CANDIDATES if p.exists()), None)

if GW_PATH is None:
    print("\nWARNING: No known groundwater file was found.")
    print("Check 01_file_inventory.csv and update GW_CANDIDATES.")
else:
    print("\nGroundwater file selected:", GW_PATH)

# ---------------------------
# 5. Audit climate files
# ---------------------------
climate_rows = []
climate_frames = {}
climate_errors = []

for path in CLIMATE_CANDIDATES:
    if not path.exists():
        continue

    try:
        df = read_table(path)
        df.columns = [str(c).strip() for c in df.columns]

        date_col = find_col(
            df, ["date", "time", "datetime", "day", "observation_date"]
        )

        parsed = parse_dates(df[date_col]) if date_col else None

        # A coordinate-wide file may contain many grid columns.
        non_date_cols = [
            c for c in df.columns if c != date_col
        ]

        climate_frames[path.name] = df

        date_min = parsed.min() if parsed is not None else pd.NaT
        date_max = parsed.max() if parsed is not None else pd.NaT

        climate_rows.append({
            "file": str(path),
            "filename": path.name,
            "rows": len(df),
            "columns": len(df.columns),
            "date_column": str(date_col) if date_col else None,
            "date_parse_success_n": int(parsed.notna().sum())
                if parsed is not None else 0,
            "date_parse_failure_n": int(parsed.isna().sum())
                if parsed is not None else len(df),
            "start_date": str(date_min) if pd.notna(date_min) else None,
            "end_date": str(date_max) if pd.notna(date_max) else None,
            "unique_dates": int(parsed.nunique())
                if parsed is not None else 0,
            "duplicate_date_rows": int(parsed.duplicated().sum())
                if parsed is not None else np.nan,
            "non_date_columns": len(non_date_cols),
            "column_names_sample": " | ".join(
                map(str, list(df.columns[:12]))
            ),
            "likely_temporal_format": (
                "date-indexed table" if date_col
                else "date column not identified"
            ),
            "status": "READ_OK"
        })

    except Exception as e:
        climate_errors.append({
            "file": str(path), "error": repr(e)
        })

climate_audit = pd.DataFrame(climate_rows)
climate_audit.to_csv(OUT / "02_climate_file_audit.csv", index=False)
pd.DataFrame(climate_errors).to_csv(
    OUT / "03_climate_read_errors.csv", index=False
)

print("\nCLIMATE FILE AUDIT")
if not climate_audit.empty:
    display(climate_audit)
else:
    print("No climate files were successfully read.")

# ---------------------------
# 6. Check date gaps for daily climate files
# ---------------------------
gap_rows = []

for filename, df in climate_frames.items():
    date_col = find_col(
        df, ["date", "time", "datetime", "day", "observation_date"]
    )
    if not date_col:
        continue

    dates = parse_dates(df[date_col]).dropna().dt.normalize()
    if dates.empty:
        continue

    unique_dates = pd.DatetimeIndex(sorted(dates.unique()))
    if len(unique_dates) < 2:
        missing_days = np.nan
        median_interval_days = np.nan
    else:
        intervals = np.diff(unique_dates.values).astype(
            "timedelta64[D]"
        ).astype(int)
        median_interval_days = float(np.median(intervals))
        expected_days = (
            unique_dates[-1] - unique_dates[0]
        ).days + 1
        missing_days = max(0, expected_days - len(unique_dates))

    gap_rows.append({
        "filename": filename,
        "unique_dates": len(unique_dates),
        "first_date": str(unique_dates.min().date()),
        "last_date": str(unique_dates.max().date()),
        "median_interval_days": median_interval_days,
        "missing_calendar_days_if_daily": missing_days,
        "interpretation_note":
            "Daily-gap estimate only; verify expected frequency first."
    })

gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(OUT / "04_climate_date_gaps.csv", index=False)

print("\nDATE COVERAGE / GAP CHECK")
display(gap_df)

# ---------------------------
# 7. Audit groundwater observations
# ---------------------------
gw_summary = {}
gw_df = None

if GW_PATH is not None:
    try:
        gw_df = read_table(GW_PATH)
        gw_df.columns = [str(c).strip() for c in gw_df.columns]

        date_col = find_col(
            gw_df, ["date", "time", "datetime", "observation_date"]
        )
        year_col = find_col(gw_df, ["year"])
        season_col = find_col(gw_df, ["season"])
        level_col = find_col(gw_df, [
            "depth_to_water_bgl_m",
            "groundwater_level_bgl",
            "water_level_bgl",
            "dtw", "water_level", "groundwater_level"
        ])
        lat_col = find_col(gw_df, ["latitude", "lat"])
        lon_col = find_col(gw_df, ["longitude", "lon", "long"])
        block_col = find_col(gw_df, ["block", "block_name", "tehsil"])
        well_col = find_col(gw_df, [
            "well_key", "well_id", "well_code", "station_id"
        ])

        if date_col:
            gw_dates = parse_dates(gw_df[date_col])
            date_min = gw_dates.min()
            date_max = gw_dates.max()
        elif year_col:
            years = pd.to_numeric(gw_df[year_col], errors="coerce")
            date_min = years.min()
            date_max = years.max()
        else:
            gw_dates = None
            date_min = date_max = np.nan

        gw_summary = {
            "file": str(GW_PATH),
            "rows": len(gw_df),
            "columns": len(gw_df.columns),
            "column_names": list(gw_df.columns),
            "date_column": str(date_col) if date_col else None,
            "year_column": str(year_col) if year_col else None,
            "season_column": str(season_col) if season_col else None,
            "water_level_column": str(level_col) if level_col else None,
            "latitude_column": str(lat_col) if lat_col else None,
            "longitude_column": str(lon_col) if lon_col else None,
            "block_column": str(block_col) if block_col else None,
            "well_id_column": str(well_col) if well_col else None,
            "start_date_or_year": (
                str(date_min) if pd.notna(date_min) else None
            ),
            "end_date_or_year": (
                str(date_max) if pd.notna(date_max) else None
            ),
            "n_unique_blocks": int(gw_df[block_col].nunique())
                if block_col else None,
            "n_unique_wells": int(gw_df[well_col].nunique())
                if well_col else None,
            "n_missing_water_level": int(gw_df[level_col].isna().sum())
                if level_col else None,
            "n_missing_latitude": int(gw_df[lat_col].isna().sum())
                if lat_col else None,
            "n_missing_longitude": int(gw_df[lon_col].isna().sum())
                if lon_col else None,
        }

        # Keep this report machine-readable
        with open(OUT / "05_groundwater_audit.json", "w") as f:
            json.dump(gw_summary, f, indent=2, default=str)

        print("\nGROUNDWATER AUDIT")
        print(json.dumps(gw_summary, indent=2, default=str))

    except Exception as e:
        print("Groundwater audit failed:", repr(e))

# ---------------------------
# 8. Check climate / groundwater temporal overlap
# ---------------------------
overlap_rows = []

if gw_df is not None:
    gw_years = set()

    gw_year_col = find_col(gw_df, ["year"])
    gw_date_col = find_col(
        gw_df, ["date", "time", "datetime", "observation_date"]
    )

    if gw_year_col:
        gw_years = set(
            pd.to_numeric(gw_df[gw_year_col], errors="coerce")
            .dropna().astype(int).tolist()
        )
    elif gw_date_col:
        gw_years = set(
            parse_dates(gw_df[gw_date_col]).dropna().dt.year.tolist()
        )

    for filename, df in climate_frames.items():
        cdate_col = find_col(
            df, ["date", "time", "datetime", "day", "observation_date"]
        )

        if cdate_col and gw_years:
            climate_dates = parse_dates(df[cdate_col]).dropna()
            climate_years = set(climate_dates.dt.year.tolist())
            shared = sorted(gw_years.intersection(climate_years))

            overlap_rows.append({
                "climate_file": filename,
                "groundwater_year_count": len(gw_years),
                "climate_year_count": len(climate_years),
                "overlap_year_count": len(shared),
                "overlap_start_year": min(shared) if shared else None,
                "overlap_end_year": max(shared) if shared else None,
                "overlap_years": ",".join(map(str, shared)),
                "note": (
                    "Year overlap only; seasonal alignment and "
                    "climate baseline suitability still need checking."
                )
            })
        else:
            overlap_rows.append({
                "climate_file": filename,
                "groundwater_year_count": len(gw_years),
                "climate_year_count": None,
                "overlap_year_count": None,
                "overlap_start_year": None,
                "overlap_end_year": None,
                "overlap_years": None,
                "note": "Could not compare years; date/year column missing."
            })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(OUT / "06_temporal_overlap.csv", index=False)

print("\nTEMPORAL OVERLAP")
display(overlap_df)

# ---------------------------
# 9. Inspect units / variable metadata
# ---------------------------
metadata_rows = []

for filename, df in climate_frames.items():
    for col in df.columns:
        metadata_rows.append({
            "filename": filename,
            "column_name": str(col),
            "dtype": str(df[col].dtype),
            "non_null_count": int(df[col].notna().sum()),
            "missing_count": int(df[col].isna().sum()),
            "unique_count": int(df[col].nunique(dropna=True)),
            "minimum": (
                float(pd.to_numeric(df[col], errors="coerce").min())
                if pd.to_numeric(df[col], errors="coerce").notna().any()
                else None
            ),
            "maximum": (
                float(pd.to_numeric(df[col], errors="coerce").max())
                if pd.to_numeric(df[col], errors="coerce").notna().any()
                else None
            ),
            "unit_status": "NOT VERIFIED FROM VALUES ALONE"
        })

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(OUT / "07_climate_column_diagnostics.csv", index=False)

print("\nCOLUMN DIAGNOSTICS (first 40 rows)")
display(metadata_df.head(40))

# ---------------------------
# 10. Produce a concise audit summary
# ---------------------------
summary = {
    "study_area": "Alwar, Rajasthan",
    "analysis_domain": "14 selected blocks",
    "scenario_of_interest": "SSP2-4.5 (SSP245)",
    "source_data_modified": False,
    "climate_files_found": int(len(climate_rows)),
    "climate_files_with_errors": int(len(climate_errors)),
    "groundwater_file_selected": str(GW_PATH) if GW_PATH else None,
    "groundwater_file_read": gw_df is not None,
    "important_caveat": (
        "This is a compatibility audit, not validation of climate "
        "model skill, bias correction, downscaling, or causal effects."
    ),
    "next_scientific_step": (
        "Verify units and provenance, select a defensible historical "
        "baseline, align seasonal climate aggregates with groundwater "
        "observations, then assess spatial coverage."
    )
}

with open(OUT / "08_step19a_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 65)
print("STEP 19A AUDIT FINISHED")
print("=" * 65)
print(json.dumps(summary, indent=2, default=str))
print("\nAudit files saved in:")
print(OUT)


**[orig #147]** Step 19B — headers and date sequences of climate CSVs  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE = Path("/content/drive/MyDrive/gw_framework")
OUT = BASE / "data/gwl/diagnostics/Alwar_Source_Recovery/Step_19A_Climate_Groundwater_Audit"
OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "tmax_hybrid": BASE / "data/Cmip6 indices/CMIP6 Indices/Comparison/ssp245/Tmax_Hybrid_SSP245_02112023.csv",
    "tmin_hybrid": BASE / "data/Cmip6 indices/CMIP6 Indices/Comparison/ssp245/Tmin_Hybrid_SSP245_02112023.csv",
    "rain_hybrid": BASE / "data/Cmip6 indices/CMIP6 Indices/Comparison/ssp245/PCP_Hybrid_SSP245_02112023.csv",
    "access_ssp245": BASE / "climate_forcing/csv/ACCESS-CM2_ssp245_bc_daily.csv",
    "access_ssp585": BASE / "climate_forcing/csv/ACCESS-CM2_ssp585_bc_daily.csv",
}

def read_csv_flexible(path):
    # Detect common delimiters without changing the source file.
    return pd.read_csv(path, sep=None, engine="python")

print("=" * 75)
print("STEP 19B — CLIMATE INPUT AND DATE-ALIGNMENT DIAGNOSTICS")
print("=" * 75)

# 1. Inspect file existence, headers and first three rows.
for name, path in FILES.items():
    print(f"\n--- {name} ---")
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    try:
        df = read_csv_flexible(path)
        print("Shape:", df.shape)
        print("First 12 columns:", df.columns[:12].tolist())
        print("First 3 rows:")
        display(df.head(3))
    except Exception as e:
        print("READ ERROR:", repr(e))

# 2. Check date sequences in the files with explicit date columns.
date_sequences = {}
for name in ["tmax_hybrid", "access_ssp245", "access_ssp585"]:
    path = FILES[name]
    if not path.exists():
        continue

    df = read_csv_flexible(path)
    date_col = next(
        (c for c in df.columns if str(c).strip().lower()
         in ["date", "time", "datetime"]),
        None
    )
    if date_col is None:
        print(f"\n{name}: no explicit date/time column detected.")
        continue

    dates = pd.to_datetime(df[date_col], errors="coerce").dropna()
    date_sequences[name] = pd.DatetimeIndex(dates).normalize()

    print(
        f"\n{name}: date column={date_col!r}; "
        f"valid={len(dates):,}; "
        f"first={dates.min()}; last={dates.max()}; "
        f"duplicate_dates={dates.dt.normalize().duplicated().sum()}"
    )

if "tmax_hybrid" in date_sequences and "access_ssp245" in date_sequences:
    same = date_sequences["tmax_hybrid"].equals(
        date_sequences["access_ssp245"]
    )
    print("\nTmax and ACCESS SSP245 date sequences identical:", same)

# 3. Diagnose the Tmin hybrid file's first column and possible date recovery.
path = FILES["tmin_hybrid"]
if path.exists():
    tmin = read_csv_flexible(path)
    print("\n--- Tmin first-column diagnosis ---")
    print("Column names:", tmin.columns[:10].tolist())
    print("First-column sample:", tmin.iloc[:10, 0].tolist())
    print("Index sample:", tmin.index[:10].tolist())

    # Check whether the first column looks like dates or years.
    first = tmin.iloc[:, 0].astype(str).str.strip()
    parsed = pd.to_datetime(first, errors="coerce")
    print("First-column date parse success:", int(parsed.notna().sum()))
    print("First-column unique values:", int(first.nunique()))

    # Compare row count with the expected daily dates, but do not assign
    # dates based on row count alone.
    expected_days = len(pd.date_range("2015-01-01", "2100-12-31", freq="D"))
    print("Rows:", len(tmin), "| Expected daily dates:", expected_days)
    print("Important: matching row counts alone do not prove row alignment.")

# 4. Inspect ranges in ACCESS-CM2 daily forcing.
for name in ["access_ssp245", "access_ssp585"]:
    path = FILES[name]
    if not path.exists():
        continue

    df = read_csv_flexible(path)
    print(f"\n--- Numeric range check: {name} ---")

    for col in df.columns:
        if str(col).strip().lower() in ["time", "date", "datetime"]:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        if values.notna().any():
            print(
                f"{col}: n={values.notna().sum():,}, "
                f"min={values.min():.5g}, "
                f"median={values.median():.5g}, "
                f"max={values.max():.5g}"
            )

# 5. Search likely directories for rainfall / precipitation data.
search_roots = [
    BASE / "data/Cmip6 indices",
    BASE / "climate_forcing",
    BASE / "data/imd",
]

print("\n--- Candidate rainfall / precipitation files ---")
found = []
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.is_file() and any(
            term in p.name.lower()
            for term in ["pcp", "precip", "rain", "pr_"]
        ):
            found.append(str(p))

for p in sorted(set(found))[:100]:
    print(p)

# 6. Save a compact diagnostic summary.
summary = {
    "files": {
        name: {
            "path": str(path),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else None
        }
        for name, path in FILES.items()
    },
    "date_sequence_comparison": {
        "tmax_vs_access_ssp245_identical": (
            date_sequences["tmax_hybrid"].equals(
                date_sequences["access_ssp245"]
            )
            if "tmax_hybrid" in date_sequences
            and "access_ssp245" in date_sequences
            else None
        )
    },
    "candidate_rainfall_files": sorted(set(found))[:100],
    "source_files_modified": False,
    "interpretation": (
        "Diagnostics only. Do not infer units, dates for Tmin, "
        "or rainfall suitability without source metadata."
    )
}

summary_path = OUT / "09_step19b_input_diagnostics.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\nSaved:", summary_path)
print("STEP 19B DIAGNOSTICS COMPLETE")


**[orig #148]** Step 19C — representative historical/SSP245 precipitation files  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
import xarray as xr

BASE = Path("/content/drive/MyDrive/gw_framework")
CLIMATE = BASE / "climate_forcing/nex_raw/ACCESS-CM2"
OUT = BASE / (
    "data/gwl/diagnostics/Alwar_Source_Recovery/"
    "Step_19A_Climate_Groundwater_Audit"
)
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("STEP 19C — NETCDF METADATA AND SCENARIO COVERAGE")
print("=" * 72)

# Inspect representative files from historical and SSP245 precipitation.
samples = [
    CLIMATE / "historical/pr/pr_2014.nc",
    CLIMATE / "ssp245/pr/pr_2015.nc",
    CLIMATE / "ssp245/pr/pr_2020.nc",
    CLIMATE / "ssp245/pr/pr_2050.nc",
    CLIMATE / "ssp245/pr/pr_2100.nc",
]

metadata_records = []

for path in samples:
    print("\n" + "-" * 72)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    try:
        with xr.open_dataset(path) as ds:
            print("Dimensions:", dict(ds.sizes))
            print("Variables:", list(ds.data_vars))
            print("Coordinates:", list(ds.coords))
            print("Global attributes:", dict(ds.attrs))

            for var in ds.data_vars:
                da = ds[var]
                print(f"\nVariable: {var}")
                print("  Dimensions:", da.dims)
                print("  Shape:", da.shape)
                print("  Attributes:", dict(da.attrs))

            if "time" in ds.coords:
                times = pd.to_datetime(ds["time"].values)
                print("Time count:", len(times))
                print("First time:", times.min())
                print("Last time:", times.max())
                print("Unique timestamps:", len(pd.Index(times).unique()))

            for coord in ["lat", "latitude", "lon", "longitude"]:
                if coord in ds.coords:
                    vals = np.asarray(ds[coord].values)
                    if vals.size:
                        print(
                            f"{coord}: min={np.nanmin(vals)}, "
                            f"max={np.nanmax(vals)}, count={vals.size}"
                        )

            for var in ds.data_vars:
                da = ds[var]
                if np.issubdtype(da.dtype, np.number):
                    metadata_records.append({
                        "file": str(path),
                        "variable": var,
                        "units": da.attrs.get("units"),
                        "standard_name": da.attrs.get("standard_name"),
                        "long_name": da.attrs.get("long_name"),
                        "dimensions": str(da.dims),
                        "shape": str(da.shape),
                        "calendar": (
                            ds["time"].encoding.get("calendar")
                            if "time" in ds.coords else None
                        ),
                    })

    except Exception as e:
        print("ERROR:", repr(e))

# Inventory annual precipitation files without opening every file.
coverage_records = []

for scenario, folder, start_year, end_year in [
    ("historical", CLIMATE / "historical/pr", 1985, 2014),
    ("ssp245", CLIMATE / "ssp245/pr", 2015, 2100),
]:
    print(f"\n--- Annual file inventory: {scenario} ---")
    existing_years = []

    for year in range(start_year, end_year + 1):
        path = folder / f"pr_{year}.nc"
        exists = path.exists()

        if exists:
            existing_years.append(year)

        coverage_records.append({
            "scenario": scenario,
            "year": year,
            "expected_file": str(path),
            "exists": exists,
        })

    missing_years = sorted(set(range(start_year, end_year + 1))
                           - set(existing_years))

    print("Expected years:", end_year - start_year + 1)
    print("Files found:", len(existing_years))
    print("Missing years:", missing_years)
    print("First available years:", existing_years[:5])
    print("Last available years:", existing_years[-5:])

metadata_df = pd.DataFrame(metadata_records)
coverage_df = pd.DataFrame(coverage_records)

metadata_path = OUT / "10_step19c_netcdf_metadata.csv"
coverage_path = OUT / "11_step19c_rainfall_file_coverage.csv"

metadata_df.to_csv(metadata_path, index=False)
coverage_df.to_csv(coverage_path, index=False)

print("\nSaved metadata:", metadata_path)
print("Saved file coverage:", coverage_path)
print("\nSTEP 19C COMPLETE — SOURCE FILES WERE NOT MODIFIED")


**[orig #149]** Step 19C — NetCDF metadata and rainfall coverage  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 19C: NETCDF METADATA + RAINFALL COVERAGE AUDIT
# Study area: Alwar, Rajasthan — 14 selected blocks
# Scenario: ACCESS-CM2 SSP2-4.5
# Source data are read-only; outputs saved separately.
# ================================================================

from pathlib import Path
import json
import warnings
import pandas as pd
import numpy as np

# -------------------- 1. Mount Google Drive ----------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# -------------------- 2. Install/check xarray -------------------
try:
    import xarray as xr
except ImportError:
    %pip -q install xarray netCDF4
    import xarray as xr

# -------------------- 3. Define paths ---------------------------
BASE = Path("/content/drive/MyDrive/gw_framework")
CLIMATE = BASE / "climate_forcing/nex_raw/ACCESS-CM2"

OUT = (
    BASE
    / "data/gwl/diagnostics/Alwar_Source_Recovery"
    / "Step_19A_Climate_Groundwater_Audit"
    / "Step_19C_NetCDF_Metadata_Audit"
)
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 78)
print("STEP 19C — NETCDF METADATA AND RAINFALL COVERAGE AUDIT")
print("=" * 78)
print("Climate directory:", CLIMATE)
print("Output directory:", OUT)
print("Original datasets will not be edited.")

# -------------------- 4. Inventory yearly files ----------------
coverage_rows = []

scenarios = [
    ("historical", CLIMATE / "historical" / "pr", 1985, 2014),
    ("ssp245", CLIMATE / "ssp245" / "pr", 2015, 2100),
]

for scenario, folder, start_year, end_year in scenarios:
    expected_years = list(range(start_year, end_year + 1))
    found_years = []

    for year in expected_years:
        file_path = folder / f"pr_{year}.nc"
        exists = file_path.is_file()

        if exists:
            found_years.append(year)

        coverage_rows.append({
            "scenario": scenario,
            "year": year,
            "file_path": str(file_path),
            "file_exists": exists,
            "file_size_MB": (
                round(file_path.stat().st_size / 1e6, 3)
                if exists else np.nan
            ),
        })

    missing_years = sorted(set(expected_years) - set(found_years))

    print(f"\n--- {scenario.upper()} file coverage ---")
    print("Expected years:", len(expected_years))
    print("Files found:", len(found_years))
    print("Missing years:", missing_years)
    print("First available years:", found_years[:5])
    print("Last available years:", found_years[-5:])

coverage_df = pd.DataFrame(coverage_rows)
coverage_csv = OUT / "01_step19c_annual_file_coverage.csv"
coverage_df.to_csv(coverage_csv, index=False)

# -------------------- 5. Select representative files -----------
sample_files = [
    CLIMATE / "historical" / "pr" / "pr_1985.nc",
    CLIMATE / "historical" / "pr" / "pr_2000.nc",
    CLIMATE / "historical" / "pr" / "pr_2014.nc",
    CLIMATE / "ssp245" / "pr" / "pr_2015.nc",
    CLIMATE / "ssp245" / "pr" / "pr_2020.nc",
    CLIMATE / "ssp245" / "pr" / "pr_2050.nc",
    CLIMATE / "ssp245" / "pr" / "pr_2100.nc",
]

metadata_rows = []
time_rows = []
coordinate_rows = []
value_rows = []
errors = []

# -------------------- 6. Inspect NetCDF metadata ---------------
for file_path in sample_files:
    print("\n" + "-" * 78)
    print("FILE:", file_path)
    print("EXISTS:", file_path.exists())

    if not file_path.exists():
        continue

    try:
        with xr.open_dataset(file_path) as ds:
            print("Dimensions:", dict(ds.sizes))
            print("Data variables:", list(ds.data_vars))
            print("Coordinates:", list(ds.coords))
            print("Global attributes:", dict(ds.attrs))

            # Record data variable metadata
            for var in ds.data_vars:
                da = ds[var]

                metadata_rows.append({
                    "file": str(file_path),
                    "variable": var,
                    "dimensions": str(da.dims),
                    "shape": str(da.shape),
                    "dtype": str(da.dtype),
                    "units": da.attrs.get("units"),
                    "standard_name": da.attrs.get("standard_name"),
                    "long_name": da.attrs.get("long_name"),
                    "cell_methods": da.attrs.get("cell_methods"),
                    "grid_mapping": da.attrs.get("grid_mapping"),
                    "attributes": json.dumps(
                        dict(da.attrs), default=str
                    ),
                })

                print(f"\nVariable {var!r}:")
                print("  Dimensions:", da.dims)
                print("  Shape:", da.shape)
                print("  Attributes:", dict(da.attrs))

                # Load one variable for basic numerical diagnostics
                if np.issubdtype(da.dtype, np.number):
                    values = np.asarray(da.values)
                    finite = values[np.isfinite(values)]

                    value_rows.append({
                        "file": str(file_path),
                        "variable": var,
                        "finite_value_count": int(finite.size),
                        "missing_value_count": int(values.size - finite.size),
                        "minimum": (
                            float(finite.min()) if finite.size else np.nan
                        ),
                        "median": (
                            float(np.median(finite))
                            if finite.size else np.nan
                        ),
                        "maximum": (
                            float(finite.max()) if finite.size else np.nan
                        ),
                        "units_metadata": da.attrs.get("units"),
                    })

            # Time metadata
            if "time" in ds.coords:
                time_coord = ds["time"]
                times = pd.to_datetime(
                    time_coord.values, errors="coerce"
                )
                times = pd.DatetimeIndex(times).dropna()

                print("\nTime coordinate:")
                print("  Count:", len(times))
                print("  First:", times.min() if len(times) else None)
                print("  Last:", times.max() if len(times) else None)
                print("  Unique timestamps:", len(times.unique()))
                print("  Encoding:", time_coord.encoding)
                print("  Attributes:", dict(time_coord.attrs))

                time_rows.append({
                    "file": str(file_path),
                    "time_count": len(times),
                    "first_time": str(times.min()) if len(times) else None,
                    "last_time": str(times.max()) if len(times) else None,
                    "unique_timestamps": len(times.unique()),
                    "time_units": time_coord.encoding.get("units"),
                    "calendar": time_coord.encoding.get("calendar"),
                })
            else:
                print("\nWARNING: No 'time' coordinate found.")
                time_rows.append({
                    "file": str(file_path),
                    "time_count": None,
                    "first_time": None,
                    "last_time": None,
                    "unique_timestamps": None,
                    "time_units": None,
                    "calendar": None,
                })

            # Spatial coordinate metadata
            for coord_name in ds.coords:
                coord = ds[coord_name]

                if coord_name.lower() in [
                    "lat", "latitude", "lon", "longitude",
                    "x", "y"
                ]:
                    vals = np.asarray(coord.values)
                    numeric = vals[
                        np.isfinite(vals)
                    ] if np.issubdtype(vals.dtype, np.number) else []

                    coordinate_rows.append({
                        "file": str(file_path),
                        "coordinate": coord_name,
                        "dimensions": str(coord.dims),
                        "count": int(vals.size),
                        "minimum": (
                            float(np.min(numeric))
                            if len(numeric) else None
                        ),
                        "maximum": (
                            float(np.max(numeric))
                            if len(numeric) else None
                        ),
                        "units": coord.attrs.get("units"),
                        "standard_name": coord.attrs.get("standard_name"),
                        "attributes": json.dumps(
                            dict(coord.attrs), default=str
                        ),
                    })

    except Exception as exc:
        errors.append({
            "file": str(file_path),
            "error": repr(exc),
        })
        print("ERROR reading file:", repr(exc))

# -------------------- 7. Save audit tables ----------------------
metadata_df = pd.DataFrame(metadata_rows)
time_df = pd.DataFrame(time_rows)
coords_df = pd.DataFrame(coordinate_rows)
values_df = pd.DataFrame(value_rows)
errors_df = pd.DataFrame(errors)

outputs = {
    "02_step19c_variable_metadata.csv": metadata_df,
    "03_step19c_time_coverage.csv": time_df,
    "04_step19c_spatial_coordinates.csv": coords_df,
    "05_step19c_value_diagnostics.csv": values_df,
    "06_step19c_read_errors.csv": errors_df,
}

for filename, df in outputs.items():
    df.to_csv(OUT / filename, index=False)

# -------------------- 8. Summary JSON ---------------------------
summary = {
    "study_area": "Alwar, Rajasthan",
    "analysis_domain": "14 selected blocks",
    "model": "ACCESS-CM2",
    "scenario": "SSP2-4.5",
    "historical_expected_years": 30,
    "ssp245_expected_years": 86,
    "historical_files_found": int(
        coverage_df.query("scenario == 'historical'")["file_exists"].sum()
    ),
    "ssp245_files_found": int(
        coverage_df.query("scenario == 'ssp245'")["file_exists"].sum()
    ),
    "historical_missing_years": coverage_df.loc[
        (coverage_df["scenario"] == "historical")
        & (~coverage_df["file_exists"]), "year"
    ].astype(int).tolist(),
    "ssp245_missing_years": coverage_df.loc[
        (coverage_df["scenario"] == "ssp245")
        & (~coverage_df["file_exists"]), "year"
    ].astype(int).tolist(),
    "representative_files_read": len(time_rows),
    "read_error_count": len(errors),
    "source_files_modified": False,
    "interpretation_caveat": (
        "Metadata audit only. Units and variable meaning must be "
        "interpreted using source metadata and processing documentation. "
        "Representative annual files do not prove complete daily coverage."
    ),
}

summary_path = OUT / "07_step19c_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

# -------------------- 9. Display results ------------------------
print("\n" + "=" * 78)
print("STEP 19C SUMMARY")
print("=" * 78)
print(json.dumps(summary, indent=2))

print("\nVARIABLE METADATA")
display(metadata_df)

print("\nTIME COVERAGE")
display(time_df)

print("\nSPATIAL COORDINATES")
display(coords_df)

print("\nVALUE DIAGNOSTICS")
display(values_df)

print("\nREAD ERRORS")
display(errors_df)

print("\nAll audit outputs saved to:")
print(OUT)
print("\nSTEP 19C COMPLETE — ORIGINAL DATASETS PRESERVED")


**[orig #150]** Step 19D — climate grid cells in an Alwar search window  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr
import json

BASE = Path("/content/drive/MyDrive/gw_framework")
SEARCH_ROOTS = [
    BASE / "climate_forcing",
    BASE / "data",
]

OUT = (
    BASE / "data/gwl/diagnostics/Alwar_Source_Recovery"
    / "Step_19D_Alwar_Grid_Search"
)
OUT.mkdir(parents=True, exist_ok=True)

# Approximate Alwar district search window; final extraction must use
# the verified 14-block polygons.
ALWAR_LAT_MIN, ALWAR_LAT_MAX = 27.0, 28.5
ALWAR_LON_MIN, ALWAR_LON_MAX = 76.0, 77.8

def find_coord(ds, candidates):
    for name in candidates:
        if name in ds.coords:
            return name
        if name in ds.variables and ds[name].ndim <= 2:
            return name
    return None

records = []
seen = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        print("Directory not found:", root)
        continue

    print("Searching:", root)
    for path in root.rglob("*.nc"):
        pstr = str(path)
        if pstr in seen:
            continue
        seen.add(pstr)

        try:
            with xr.open_dataset(path, decode_times=False) as ds:
                lat_name = find_coord(
                    ds, ["lat", "latitude", "y"]
                )
                lon_name = find_coord(
                    ds, ["lon", "longitude", "x"]
                )

                if lat_name is None or lon_name is None:
                    continue

                lat = np.asarray(ds[lat_name].values, dtype=float)
                lon = np.asarray(ds[lon_name].values, dtype=float)

                if lat.size == 0 or lon.size == 0:
                    continue

                # Only handle standard 1-D latitude/longitude here.
                # Curvilinear grids require a separate spatial check.
                if lat.ndim != 1 or lon.ndim != 1:
                    continue

                lat_min, lat_max = np.nanmin(lat), np.nanmax(lat)
                lon_min, lon_max = np.nanmin(lon), np.nanmax(lon)

                overlaps_alwar = (
                    lat_max >= ALWAR_LAT_MIN
                    and lat_min <= ALWAR_LAT_MAX
                    and lon_max >= ALWAR_LON_MIN
                    and lon_min <= ALWAR_LON_MAX
                )

                variables = list(ds.data_vars)
                time_count = (
                    int(ds.sizes["time"])
                    if "time" in ds.sizes else None
                )

                records.append({
                    "file": pstr,
                    "variables": ",".join(variables),
                    "latitude_min": float(lat_min),
                    "latitude_max": float(lat_max),
                    "longitude_min": float(lon_min),
                    "longitude_max": float(lon_max),
                    "n_lat": int(lat.size),
                    "n_lon": int(lon.size),
                    "time_count": time_count,
                    "overlaps_approx_alwar": overlaps_alwar,
                    "scenario": ds.attrs.get("scenario"),
                    "source_id": ds.attrs.get("cmip6_source_id"),
                    "activity": ds.attrs.get("activity"),
                })

        except Exception:
            # Skip files that cannot be opened as standard NetCDF.
            continue

df = pd.DataFrame(records)

if df.empty:
    print("No standard 1-D latitude/longitude NetCDF grids found.")
else:
    df = df.drop_duplicates(subset=["file"]).sort_values(
        ["overlaps_approx_alwar", "file"],
        ascending=[False, True]
    )

    output_csv = OUT / "01_netcdf_grid_coverage_inventory.csv"
    df.to_csv(output_csv, index=False)

    matches = df[df["overlaps_approx_alwar"]]
    match_csv = OUT / "02_candidate_files_covering_alwar.csv"
    matches.to_csv(match_csv, index=False)

    print("\nNetCDF files with readable 1-D grids:", len(df))
    print("Files overlapping approximate Alwar window:", len(matches))

    print("\nCandidate files:")
    display(matches.head(50))

    print("\nCurrent ACCESS-CM2 precipitation grid:")
    display(
        df[df["file"].str.contains(
            "ACCESS-CM2", case=False, na=False
        )].head(20)
    )

    print("\nSaved full inventory:", output_csv)
    print("Saved Alwar candidates:", match_csv)

print("\nSTEP 19D GRID SEARCH COMPLETE")


**[orig #151]** Step 19E — source/scenario/variable metadata review table  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import os

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_19D_Alwar_Grid_Search"
)

INV = BASE / "01_netcdf_grid_coverage_inventory.csv"
CAND = BASE / "02_candidate_files_covering_alwar.csv"

inventory = pd.read_csv(INV)
candidates = pd.read_csv(CAND)

print("Inventory rows:", len(inventory))
print("Alwar-overlapping candidate rows:", len(candidates))
print("\nColumns:")
print(inventory.columns.tolist())

# Show source, scenario and variable metadata where available
for col in ["source_id", "scenario", "activity", "variables"]:
    if col in candidates.columns:
        print(f"\n--- {col}: counts ---")
        print(candidates[col].fillna("UNKNOWN").value_counts().head(30))

# Display complete paths rather than truncated notebook output
path_col = "file" if "file" in candidates.columns else candidates.columns[0]
show_cols = [
    c for c in [
        path_col, "variables", "latitude_min", "latitude_max",
        "longitude_min", "longitude_max", "n_lat", "n_lon",
        "time_count", "scenario", "source_id", "activity"
    ] if c in candidates.columns
]

print("\n--- Candidate paths and metadata ---")
with pd.option_context(
    "display.max_rows", 250,
    "display.max_colwidth", 180,
    "display.width", 240
):
    display(candidates[show_cols])

# Save a simplified review table
review_path = BASE / "03_step19e_candidate_review.csv"
candidates[show_cols].to_csv(review_path, index=False)
print("\nSaved review table:", review_path)


**[orig #152]** Step 19F — find climate-projection files  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import re

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_19D_Alwar_Grid_Search"
)
inv = pd.read_csv(BASE / "01_netcdf_grid_coverage_inventory.csv")

# Find files whose paths or metadata suggest climate-model projections
path = inv["file"].fillna("").astype(str)
text_cols = [c for c in ["variables", "scenario", "source_id", "activity"]
             if c in inv.columns]
meta_text = inv[text_cols].fillna("").astype(str).agg(" ".join, axis=1)

mask = (
    path.str.contains(
        r"cmip|nex|gddp|access|ssp245|ssp2.?4.?5|historical",
        case=False, regex=True
    )
    | meta_text.str.contains(
        r"cmip|nex|gddp|access|ssp245|ssp2.?4.?5|historical",
        case=False, regex=True
    )
)
models = inv.loc[mask].copy()

print("Likely CMIP6/model files in inventory:", len(models))
print("\nVariable signatures:")
print(models["variables"].fillna("UNKNOWN").value_counts().head(40))

print("\nScenario metadata:")
print(models["scenario"].fillna("UNKNOWN").value_counts())

print("\nModel metadata:")
print(models["source_id"].fillna("UNKNOWN").value_counts())

cols = [
    c for c in [
        "file", "variables", "latitude_min", "latitude_max",
        "longitude_min", "longitude_max", "time_count",
        "scenario", "source_id", "activity", "overlaps_approx_alwar"
    ] if c in models.columns
]

# Prioritize files whose filenames indicate SSP245 or historical data
models["filename_priority"] = path.loc[models.index].str.contains(
    r"ssp245|ssp2.?4.?5|historical",
    case=False, regex=True
)
models = models.sort_values("filename_priority", ascending=False)

with pd.option_context("display.max_rows", 200,
                      "display.max_colwidth", 180,
                      "display.width", 240):
    display(models[cols].head(200))

out = BASE / "04_step19f_likely_cmip6_files.csv"
models[cols].to_csv(out, index=False)
print("\nSaved:", out)


**[orig #153]** Install download packages  
<sub>✅ ran OK</sub>

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

!pip -q install requests beautifulsoup4 xarray netCDF4 pandas


**[orig #154]** Step 20A — NASA NEX-GDDP-CMIP6 subsets for Alwar (explored)  
<sub>✅ ran OK</sub>

In [ ]:

import os
import re
import time
import json
import requests
import pandas as pd
import xarray as xr
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote

# ============================================================
# STEP 20A — Download NASA NEX-GDDP-CMIP6 spatial subsets
# Study area: Alwar, Rajasthan, India
# ============================================================

# ---- Configuration ----
MODEL = "ACCESS-CM2"
SCENARIO = "ssp245"
VARIANTS = "r1i1p1f1"

VARIABLES = ["pr", "tasmax", "tasmin"]
YEARS = range(2015, 2101)

# Approximate Alwar-region bounding box.
# These bounds are intentionally broader than the district.
LAT_MIN, LAT_MAX = 27.0, 28.5
LON_MIN, LON_MAX = 76.0, 77.8

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Alwar_NEX_GDDP_CMIP6/"
    f"{MODEL}_{SCENARIO}"
)
os.makedirs(OUT_DIR, exist_ok=True)

BASE = "https://ds.nccs.nasa.gov/thredds"
CATALOG_BASE = (
    f"{BASE}/catalog/AMES/NEX/GDDP-CMIP6/"
    f"{MODEL}/{SCENARIO}/{VARIANTS}"
)
NCSS_BASE = (
    f"{BASE}/ncss/grid/AMES/NEX/GDDP-CMIP6/"
    f"{MODEL}/{SCENARIO}/{VARIANTS}"
)

session = requests.Session()
session.headers.update({"User-Agent": "Alwar-PhD-CMIP6-subsetting/1.0"})

inventory = []
errors = []

def get_year_file(variable, year):
    """
    Discover the actual yearly NetCDF filename from NASA's catalog.
    This avoids assuming a particular file-version suffix.
    """
    catalog_url = f"{CATALOG_BASE}/{variable}/catalog.html"
    response = session.get(catalog_url, timeout=90)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    candidates = []

    for a in soup.find_all("a", href=True):
        href = unquote(a["href"])
        name = os.path.basename(href.split("?")[0])

        if (
            name.endswith(".nc")
            and f"{year}" in name
            and name.startswith(f"{variable}_day_{MODEL}_{SCENARIO}_")
        ):
            candidates.append(name)

    # Prefer the catalog's versioned file if duplicates are listed.
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            f"No yearly file found in catalog: "
            f"{variable}, {year}, {catalog_url}"
        )

    return candidates[-1]

def download_subset(variable, year):
    filename = get_year_file(variable, year)

    dataset_path = (
        f"AMES/NEX/GDDP-CMIP6/{MODEL}/{SCENARIO}/"
        f"{VARIANTS}/{variable}/{filename}"
    )

    # NASA THREDDS NetCDF Subset Service (NCSS)
    ncss_url = f"{NCSS_BASE}/{variable}/{filename}"

    params = {
        "var": variable,
        "north": LAT_MAX,
        "south": LAT_MIN,
        "west": LON_MIN,
        "east": LON_MAX,
        "horizStride": 1,
        "time_start": f"{year}-01-01T12:00:00Z",
        "time_end": f"{year}-12-31T12:00:00Z",
        "accept": "netcdf4",
        "addLatLon": "true",
    }

    outfile = os.path.join(
        OUT_DIR, f"{variable}_{MODEL}_{SCENARIO}_{year}_Alwar.nc"
    )

    if os.path.exists(outfile) and os.path.getsize(outfile) > 1000:
        return outfile, "EXISTS"

    response = session.get(
        ncss_url, params=params, timeout=(30, 300), stream=True
    )
    response.raise_for_status()

    temp_file = outfile + ".part"
    with open(temp_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    if os.path.getsize(temp_file) < 1000:
        os.remove(temp_file)
        raise IOError("Downloaded response is unexpectedly small.")

    # Confirm the response is readable NetCDF before retaining it.
    try:
        with xr.open_dataset(temp_file) as ds:
            if variable not in ds.variables:
                raise ValueError(
                    f"Expected variable {variable}; found {list(ds.variables)}"
                )
            lat_name = next(
                (n for n in ["lat", "latitude"] if n in ds.variables), None
            )
            lon_name = next(
                (n for n in ["lon", "longitude"] if n in ds.variables), None
            )
            if lat_name and lon_name:
                lats = ds[lat_name].values
                lons = ds[lon_name].values
                if lats.min() < -90 or lats.max() > 90:
                    raise ValueError("Invalid latitude coordinates.")
                if lons.min() < 0 or lons.max() > 360:
                    raise ValueError("Unexpected longitude coordinate range.")
    except Exception:
        if os.path.exists(temp_file):
            os.remove(temp_file)
        raise

    os.replace(temp_file, outfile)
    return outfile, "DOWNLOADED"

# ---- Download loop ----
for variable in VARIABLES:
    for year in YEARS:
        try:
            path, status = download_subset(variable, year)
            inventory.append({
                "model": MODEL,
                "scenario": SCENARIO,
                "variable": variable,
                "year": year,
                "status": status,
                "file": path,
                "size_MB": round(os.path.getsize(path) / 1e6, 3),
            })
            print(f"OK: {variable} {year} — {status}")
        except Exception as e:
            errors.append({
                "variable": variable,
                "year": year,
                "error": str(e)[:500],
            })
            print(f"FAILED: {variable} {year}: {str(e)[:180]}")

        # Be courteous to the public data service.
        time.sleep(0.3)

inventory_df = pd.DataFrame(inventory)
errors_df = pd.DataFrame(errors)

inventory_path = os.path.join(OUT_DIR, "download_inventory.csv")
errors_path = os.path.join(OUT_DIR, "download_errors.csv")

inventory_df.to_csv(inventory_path, index=False)
errors_df.to_csv(errors_path, index=False)

print("\nDOWNLOAD SUMMARY")
print("Successful/existing files:", len(inventory_df))
print("Failures:", len(errors_df))
print("Output directory:", OUT_DIR)
print("Inventory:", inventory_path)
print("Errors:", errors_path)


## 2. Download and extract the Mishra CMIP6 archive

**[orig #155]** **MAIN** — download Mishra et al. CMIP6 `India.tar.xz` from Zenodo (was interrupted; archive is on Drive)  
<sub>⏸ interrupted (long run or waiting for upload)</sub>

In [ ]:

import os
import requests
from tqdm.auto import tqdm

# --------------------------------------------------
# Dataset: India.tar.xz from Zenodo
# --------------------------------------------------
URL = "https://zenodo.org/records/3987736/files/India.tar.xz?download=1"

SAVE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6"
)
FILE_NAME = "India.tar.xz"

os.makedirs(SAVE_DIR, exist_ok=True)

FINAL_PATH = os.path.join(SAVE_DIR, FILE_NAME)
TEMP_PATH = FINAL_PATH + ".part"

# --------------------------------------------------
# Download with resume support where available
# --------------------------------------------------
existing_size = (
    os.path.getsize(TEMP_PATH)
    if os.path.exists(TEMP_PATH)
    else 0
)

headers = {}
if existing_size > 0:
    headers["Range"] = f"bytes={existing_size}-"

with requests.get(
    URL,
    headers=headers,
    stream=True,
    timeout=(30, 180),
    allow_redirects=True
) as response:

    # A server may ignore the Range header and send the whole file.
    if existing_size > 0 and response.status_code == 206:
        mode = "ab"
    else:
        mode = "wb"
        existing_size = 0

    response.raise_for_status()

    total = response.headers.get("Content-Length")
    total = int(total) + existing_size if total else None

    with open(TEMP_PATH, mode) as f:
        with tqdm(
            total=total,
            initial=existing_size,
            unit="B",
            unit_scale=True,
            desc="Downloading India.tar.xz"
        ) as progress:
            for chunk in response.iter_content(
                chunk_size=4 * 1024 * 1024
            ):
                if chunk:
                    f.write(chunk)
                    progress.update(len(chunk))

# --------------------------------------------------
# Basic validation
# --------------------------------------------------
file_size = os.path.getsize(TEMP_PATH)

if file_size < 1000:
    raise RuntimeError(
        f"Downloaded file is unexpectedly small: {file_size} bytes. "
        "Keep the .part file for diagnosis."
    )

# Verify that the file is an XZ archive.
with open(TEMP_PATH, "rb") as f:
    magic = f.read(6)

if magic != bytes([0xFD, 0x37, 0x7A, 0x58, 0x5A, 0x00]):
    raise RuntimeError(
        "The downloaded file does not have a valid XZ signature. "
        "It may be an HTML error page. Inspect the response before retrying."
    )

os.replace(TEMP_PATH, FINAL_PATH)

print("\nDownload completed.")
print("File:", FINAL_PATH)
print(f"Size: {file_size / (1024**3):.3f} GiB")
print("XZ signature: valid")


**[orig #156]** List archive members without extracting  
<sub>✅ ran OK</sub>

In [ ]:

import os
import subprocess
import pandas as pd

ARCHIVE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/Copy of India.tar.xz"
)

# Mount Drive if it is not already mounted
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isfile(ARCHIVE):
    raise FileNotFoundError(
        "Archive not found. Check the Google Drive folder and filename:\n"
        + ARCHIVE
    )

print("Archive:", ARCHIVE)
print("File size (GiB):", round(os.path.getsize(ARCHIVE) / 1024**3, 2))

LISTING = os.path.join(
    os.path.dirname(ARCHIVE), "India_archive_file_listing.txt"
)

# List members without extracting them
with open(LISTING, "w") as out:
    result = subprocess.run(
        ["tar", "-tJf", ARCHIVE],
        stdout=out,
        stderr=subprocess.PIPE,
        text=True
    )

if result.returncode != 0:
    print("Archive listing error:")
    print(result.stderr[-3000:])
    raise RuntimeError("Could not list the archive. See error above.")

print("Archive listing saved:", LISTING)

# Read the listing and identify climate-related files
with open(LISTING, "r", errors="replace") as f:
    members = [line.strip() for line in f if line.strip()]

print("Total archive members:", len(members))
print("\nFirst 30 archive members:")
for name in members[:30]:
    print(name)

terms = [
    "pr_", "precip", "rain",
    "tasmax", "tasmin", "tmax", "tmin",
    "ssp245", "ssp2-4.5", "historical"
]

matches = [
    name for name in members
    if any(term in name.lower() for term in terms)
]

print("\nClimate-related filename matches:", len(matches))
for name in matches[:100]:
    print(name)

MATCH_CSV = os.path.join(
    os.path.dirname(ARCHIVE), "India_climate_filename_matches.csv"
)
pd.DataFrame({"archive_member": matches}).to_csv(
    MATCH_CSV, index=False
)
print("\nSaved filename matches:", MATCH_CSV)


**[orig #157]** Group archive members by variable  
<sub>✅ ran OK</sub>

In [ ]:

import re
import os
import pandas as pd

with open(LISTING, "r", errors="replace") as f:
    members = [line.strip() for line in f if line.strip()]

def classify_variable(name):
    s = name.lower()
    if re.search(r"(^|[/_.-])tasmax([/_.-]|$)", s):
        return "tasmax"
    if re.search(r"(^|[/_.-])tasmin([/_.-]|$)", s):
        return "tasmin"
    if re.search(r"(^|[/_.-])pr([/_.-]|$)", s):
        return "pr"
    return None

rows = []
for name in members:
    var = classify_variable(name)
    if var:
        years = re.findall(r"(?<!\d)(19\d{2}|20\d{2})(?!\d)", name)
        rows.append({
            "variable": var,
            "filename_years": ",".join(years),
            "filename": name
        })

df = pd.DataFrame(rows)

if df.empty:
    print("No standard pr/tasmax/tasmin filenames detected.")
    print("Please inspect the first 100 archive members from Step 1.")
else:
    print("Files grouped by variable:")
    display(df.groupby("variable").size().rename("file_count").to_frame())

    print("\nExample files:")
    display(df.groupby("variable", group_keys=False).head(10))

    print("\nScenario counts inferred from filenames:")
    scenario_rows = []
    for name in df["filename"]:
        s = name.lower()
        if "ssp245" in s or "ssp2-4.5" in s:
            scenario_rows.append("ssp245")
        elif "ssp585" in s or "ssp5-8.5" in s:
            scenario_rows.append("ssp585")
        elif "historical" in s:
            scenario_rows.append("historical")
        else:
            scenario_rows.append("unknown")
    df["scenario_from_filename"] = scenario_rows
    display(
        df.groupby(["variable", "scenario_from_filename"])
          .size().rename("file_count").to_frame()
    )


**[orig #158]** Fresh listing + extension counts  
<sub>✅ ran OK</sub>

In [ ]:

import os
import subprocess
import pandas as pd
from IPython.display import display

ARCHIVE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/Copy of India.tar.xz"
)

LISTING = os.path.join(
    os.path.dirname(ARCHIVE),
    "India_archive_file_listing.txt"
)

if not os.path.exists(ARCHIVE):
    raise FileNotFoundError(f"Archive not found: {ARCHIVE}")

# Generate a fresh listing
with open(LISTING, "w") as out:
    result = subprocess.run(
        ["tar", "-tJf", ARCHIVE],
        stdout=out,
        stderr=subprocess.PIPE,
        text=True
    )

if result.returncode != 0:
    raise RuntimeError(result.stderr)

with open(LISTING, "r", errors="replace") as f:
    members = [line.strip() for line in f if line.strip()]

print("Total archive members:", len(members))
print("\nFIRST 100 ARCHIVE MEMBERS\n")

for i, member in enumerate(members[:100], start=1):
    print(f"{i:03d}. {member}")

print("\nFILE EXTENSION COUNTS")
extensions = pd.Series([
    os.path.splitext(m.split("/")[-1])[1].lower() or "[no extension]"
    for m in members
]).value_counts()

display(extensions.to_frame("count"))

print("\nCLIMATE-RELATED FILES (sample)")
keywords = [
    "precip", "rain", "prcp", "tmax", "tmin",
    "tas", "temperature", "historical", "ssp",
    "netcdf", ".nc", ".nc4", ".grb", ".grib"
]

matches = [
    m for m in members
    if any(k in m.lower() for k in keywords)
]

for member in matches[:100]:
    print(member)

print("\nFull listing saved at:")
print(LISTING)


**[orig #161]** **MAIN** — extract the archive to `Vimal_Mishra_CMIP6/SSP245_extracted/India`  
<sub>✅ ran OK</sub>

In [ ]:

import os
import subprocess
import shutil

# Reuse the location of the listing created earlier
LISTING = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "India_archive_file_listing.txt"
)

# If your listing is stored elsewhere, change LISTING above.
if not os.path.isfile(LISTING):
    raise FileNotFoundError(
        "Could not find the previous archive listing. "
        "Set LISTING to its actual Google Drive path."
    )

# The archive should be in the same folder as the listing.
folder = os.path.dirname(LISTING)
candidates = [
    os.path.join(folder, "Copy of India.tar.xz"),
    os.path.join(folder, "India.tar.xz"),
]

ARCHIVE = next((p for p in candidates if os.path.isfile(p)), None)

if ARCHIVE is None:
    raise FileNotFoundError(
        "Archive not found beside the listing. "
        "Set ARCHIVE to its actual path."
    )

# Extract into a separate folder; keep the original archive.
OUT_DIR = os.path.join(folder, "SSP245_extracted")
os.makedirs(OUT_DIR, exist_ok=True)

print("Archive:", ARCHIVE)
print("Output:", OUT_DIR)
print("Available space in output filesystem (GB):",
      round(shutil.disk_usage(OUT_DIR).free / 1e9, 2))

# Extract SSP245 folders only. This is a single archive pass.
cmd = [
    "tar", "-xJf", ARCHIVE,
    "-C", OUT_DIR,
    "--wildcards",
    "India/*/ssp245",
]

print("\nExtracting SSP245 data for all models...")
result = subprocess.run(cmd, text=True, capture_output=True)

if result.returncode != 0:
    print(result.stderr[-5000:])
    raise RuntimeError("Extraction failed. See the error above.")

print("Extraction command completed.")

# Scan extracted files, not the compressed archive.
records = []
for current, dirs, files in os.walk(OUT_DIR):
    for name in files:
        path = os.path.join(current, name)
        rel = os.path.relpath(path, OUT_DIR)
        records.append({
            "relative_path": rel,
            "size_MB": round(os.path.getsize(path) / 1e6, 3),
            "size_GB": round(os.path.getsize(path) / 1e9, 4),
        })

import pandas as pd
inventory = pd.DataFrame(records)
inventory_path = os.path.join(OUT_DIR, "extracted_file_inventory.csv")
inventory.to_csv(inventory_path, index=False)

print("\nExtracted files:", len(inventory))
display(inventory.sort_values("size_MB", ascending=False))
print("Inventory saved:", inventory_path)


**[orig #163]** Confirm extracted models (13)  
<sub>✅ ran OK</sub>

In [ ]:

import os

ROOT = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6"
)

EXTRACTED = os.path.join(ROOT, "SSP245_extracted")
INDIA = os.path.join(EXTRACTED, "India")

if os.path.isdir(INDIA):
    print("✅ Extracted CMIP6 data found in Google Drive.")

    models = [
        name for name in os.listdir(INDIA)
        if os.path.isdir(os.path.join(INDIA, name, "ssp245"))
    ]

    print("Models available:", len(models))
    print("Models:", sorted(models))

    print("\nNext output directory:")
    print(os.path.join(EXTRACTED, "Alwar_grid_analysis"))
else:
    print("⚠️ Extracted folder not found at:")
    print(INDIA)
    print("Check the folder path in Google Drive.")


## 3. Grid cells and block weights

Result: 22 grid cells intersect the 14 blocks.

**[orig #164]** Read longitude/latitude header rows of each model file  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import numpy as np

ROOT = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/SSP245_extracted/India"
)

OUT_DIR = os.path.dirname(ROOT)
COORD_DIR = os.path.join(OUT_DIR, "Alwar_grid_analysis")
os.makedirs(COORD_DIR, exist_ok=True)

VARIABLES = ["PrecipData", "TMaxData", "TMinData"]
records = []

for model in sorted(os.listdir(ROOT)):
    model_dir = os.path.join(ROOT, model, "ssp245")
    if not os.path.isdir(model_dir):
        continue

    for variable in VARIABLES:
        path = os.path.join(model_dir, variable)
        if not os.path.isfile(path):
            print("Missing:", path)
            continue

        try:
            # The README says row 1 has longitude, row 2 latitude.
            # Read only these two rows, treating all columns as strings.
            header = pd.read_csv(
                path,
                sep=None,
                engine="python",
                header=None,
                nrows=2,
                dtype=str
            )

            if header.shape[0] < 2 or header.shape[1] < 4:
                print("Unexpected header structure:", path, header.shape)
                continue

            # First three columns are date fields; climate grids start at column 4.
            for col in range(3, header.shape[1]):
                lon = pd.to_numeric(header.iloc[0, col], errors="coerce")
                lat = pd.to_numeric(header.iloc[1, col], errors="coerce")

                if pd.notna(lon) and pd.notna(lat):
                    records.append({
                        "model": model,
                        "scenario": "ssp245",
                        "variable": variable,
                        "column_index_zero_based": col,
                        "longitude": float(lon),
                        "latitude": float(lat),
                        "source_file": path
                    })

            print(f"Read coordinate header: {model} / {variable}")

        except Exception as e:
            print(f"ERROR: {model} / {variable}: {e}")

coords = pd.DataFrame(records)
coord_path = os.path.join(COORD_DIR, "ssp245_grid_coordinates.csv")
coords.to_csv(coord_path, index=False)

print("\nCoordinate records:", len(coords))
print("Models:", coords["model"].nunique() if not coords.empty else 0)
print("Saved:", coord_path)

if not coords.empty:
    print("\nCoordinate extent:")
    display(coords[["longitude", "latitude"]].agg(["min", "max"]))
    print("\nExample coordinates:")
    display(coords.head(10))


**[orig #166]** Select Alwar block polygons (LGD 574–587)  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import geopandas as gpd

BASE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

COORD_FILE = os.path.join(BASE, "ssp245_grid_coordinates.csv")
BLOCK_FILE = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)

OUT_DIR = os.path.join(BASE, "Alwar_spatial_selection")
os.makedirs(OUT_DIR, exist_ok=True)

# Load files
coords = pd.read_csv(COORD_FILE)
blocks = gpd.read_file(BLOCK_FILE)

# Correct field name from your GeoJSON
LGD_FIELD = "block_lgd"

blocks[LGD_FIELD] = pd.to_numeric(
    blocks[LGD_FIELD], errors="coerce"
)

# Select Alwar block LGD codes 574–587
target_codes = list(range(574, 588))
alwar = blocks[blocks[LGD_FIELD].isin(target_codes)].copy()

print("Selected polygon features:", len(alwar))
print("\nSelected LGD codes and block names:")
display(
    alwar[[LGD_FIELD, "block_name", "district"]]
    .drop_duplicates()
    .sort_values(LGD_FIELD)
)

found_codes = sorted(alwar[LGD_FIELD].dropna().astype(int).unique())
missing_codes = sorted(set(target_codes) - set(found_codes))

if missing_codes:
    print("\nWARNING — missing LGD codes:", missing_codes)

if len(found_codes) != 14:
    raise ValueError(
        f"Expected 14 unique LGD codes, found {len(found_codes)}. "
        "Review the block-name/code table before proceeding."
    )

# Create the study-area union
alwar = alwar.to_crs("EPSG:4326")
study_area = alwar.geometry.union_all()

print("\nStudy-area bounds (min lon, min lat, max lon, max lat):")
print(study_area.bounds)

# Filter coordinate records spatially
grid = coords.drop_duplicates(
    subset=["model", "variable", "column_index_zero_based", "source_file"]
).copy()

points = gpd.GeoDataFrame(
    grid,
    geometry=gpd.points_from_xy(
        grid["longitude"], grid["latitude"]
    ),
    crs="EPSG:4326"
)

selected = points[
    points.geometry.intersects(study_area)
].copy()

selected["longitude"] = selected.geometry.x
selected["latitude"] = selected.geometry.y

selected = pd.DataFrame(
    selected.drop(columns="geometry")
)

out_csv = os.path.join(
    OUT_DIR, "Alwar_selected_grid_cells_inside_blocks.csv"
)
selected.to_csv(out_csv, index=False)

print("\nSelected coordinate records:", len(selected))
print("Models:", selected["model"].nunique())
print("\nGrid cells by model and variable:")
display(
    selected.groupby(["model", "variable"])
    .size()
    .rename("selected_grid_cells")
    .reset_index()
)

print("\nSaved:", out_csv)


**[orig #167]** Extract the grid columns covering Alwar  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import numpy as np

ROOT = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/SSP245_extracted/India"
)

BASE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

SELECTED_FILE = os.path.join(
    BASE, "Alwar_spatial_selection",
    "Alwar_selected_grid_cells_inside_blocks.csv"
)

OUT_DIR = os.path.join(BASE, "Alwar_daily_extracted")
os.makedirs(OUT_DIR, exist_ok=True)

selected = pd.read_csv(SELECTED_FILE)

# Read unique source file + column combinations
selected = selected.drop_duplicates(
    subset=["source_file", "column_index_zero_based"]
)

# Variables and date columns
DATE_COLS = [0, 1, 2]
CHUNK_SIZE = 2000

inventory = []
errors = []

for source_file, group in selected.groupby("source_file"):
    model = group["model"].iloc[0]
    variable = group["variable"].iloc[0]

    # Preserve the actual column order from the original file
    grid_columns = sorted(
        group["column_index_zero_based"].astype(int).unique().tolist()
    )
    usecols = DATE_COLS + grid_columns

    output_path = os.path.join(
        OUT_DIR, f"{model}_ssp245_{variable}_Alwar.csv"
    )

    print(f"\nExtracting {model} / {variable}")
    print("Selected grid columns:", len(grid_columns))

    # Remove an earlier partial output before restarting this file.
    if os.path.exists(output_path):
        os.remove(output_path)

    try:
        # Skip the two coordinate-header rows; data begins at row 3.
        reader = pd.read_csv(
            source_file,
            header=None,
            skiprows=2,
            usecols=usecols,
            chunksize=CHUNK_SIZE,
            sep=None,
            engine="python"
        )

        first_chunk = True
        total_rows = 0

        for chunk in reader:
            # Ensure date columns are named consistently.
            chunk.columns = [
                "Year", "Month", "Day"
            ] + [
                f"grid_col_{c}" for c in grid_columns
            ]

            # Convert dates to numeric; bad date rows will be audited later.
            for col in ["Year", "Month", "Day"]:
                chunk[col] = pd.to_numeric(
                    chunk[col], errors="coerce"
                )

            chunk.to_csv(
                output_path,
                index=False,
                mode="w" if first_chunk else "a",
                header=first_chunk
            )

            total_rows += len(chunk)
            first_chunk = False

        inventory.append({
            "model": model,
            "variable": variable,
            "source_file": source_file,
            "output_file": output_path,
            "grid_columns": len(grid_columns),
            "rows_extracted": total_rows,
            "output_MB": round(
                os.path.getsize(output_path) / 1e6, 3
            )
        })

        print("Rows extracted:", total_rows)
        print("Saved:", output_path)

    except Exception as e:
        errors.append({
            "model": model,
            "variable": variable,
            "source_file": source_file,
            "error": str(e)
        })
        print("ERROR:", repr(e))

pd.DataFrame(inventory).to_csv(
    os.path.join(OUT_DIR, "extraction_inventory.csv"),
    index=False
)
pd.DataFrame(errors).to_csv(
    os.path.join(OUT_DIR, "extraction_errors.csv"),
    index=False
)

print("\nExtraction summary")
print("Completed files:", len(inventory))
print("Failed files:", len(errors))
print("Output directory:", OUT_DIR)

display(pd.DataFrame(inventory))
if errors:
    display(pd.DataFrame(errors))


**[orig #168]** Verify model × variable files and sizes  
<sub>✅ ran OK</sub>

In [ ]:

import os
import glob
import pandas as pd
from collections import Counter

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_daily_extracted"
)

models = [
    "ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CanESM5",
    "EC-Earth3", "EC-Earth3-Veg", "INM-CM4-8", "INM-CM5-0",
    "MPI-ESM1-2-HR", "MPI-ESM1-2-LR", "MRI-ESM2-0",
    "NorESM2-LM", "NorESM2-MM"
]

variables = ["PrecipData", "TMaxData", "TMinData"]

expected = {
    f"{model}_ssp245_{variable}_Alwar.csv"
    for model in models
    for variable in variables
}

actual_paths = glob.glob(os.path.join(OUT_DIR, "*.csv"))
actual_names = [os.path.basename(p) for p in actual_paths]
actual_set = set(actual_names)

missing = sorted(expected - actual_set)
unexpected = sorted(actual_set - expected)
duplicates = sorted(
    name for name, count in Counter(actual_names).items() if count > 1
)

print("Expected climate files:", len(expected))
print("Actual CSV files in folder:", len(actual_paths))
print("Expected files found:", len(expected & actual_set))
print("Missing expected files:", len(missing))
print("Unexpected CSV files:", len(unexpected))
print("Duplicate filenames:", len(duplicates))

if missing:
    print("\nMISSING FILES:")
    print("\n".join(missing))

if unexpected:
    print("\nOTHER CSV FILES (may be audit/inventory files):")
    print("\n".join(unexpected))

# Verify model-variable combinations and file sizes
records = []
for model in models:
    for variable in variables:
        filename = f"{model}_ssp245_{variable}_Alwar.csv"
        path = os.path.join(OUT_DIR, filename)
        exists = os.path.isfile(path)

        records.append({
            "model": model,
            "variable": variable,
            "exists": exists,
            "size_MB": round(os.path.getsize(path) / (1024**2), 3)
                        if exists else None
        })

check = pd.DataFrame(records)
display(check)

print("\nModels with all 3 variable files:")
counts = check.groupby("model")["exists"].sum()
print(counts.to_string())

print("\nALL 39 EXPECTED FILES PRESENT:", len(expected & actual_set) == 39)


**[orig #170]** Date-column audit of extracted files  
<sub>✅ ran OK</sub>

In [ ]:

import os
import glob
import pandas as pd

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_daily_extracted"
)

files = sorted(glob.glob(os.path.join(OUT_DIR, "*_ssp245_*_Alwar.csv")))

results = []
for path in files:
    name = os.path.basename(path)
    size_mb = os.path.getsize(path) / (1024 ** 2)

    # Read only the date columns to keep memory use low
    dates = pd.read_csv(
        path,
        usecols=["Year", "Month", "Day"]
    )

    results.append({
        "filename": name,
        "size_MB": round(size_mb, 3),
        "rows": len(dates),
        "date_columns_present": (
            {"Year", "Month", "Day"}.issubset(dates.columns)
        )
    })

audit = pd.DataFrame(results)

expected_models = 13
expected_variables = 3
expected_files = expected_models * expected_variables

print("Expected files:", expected_files)
print("Files found:", len(audit))
print("All date columns present:", audit["date_columns_present"].all())
print("Files with zero rows:", int((audit["rows"] == 0).sum()))

print("\nRows per file:")
print(audit["rows"].value_counts().sort_index().to_string())

print("\nModel count:", audit["filename"].str.split("_ssp245_").str[0].nunique())
print("\nVariable counts:")
print(
    audit["filename"]
    .str.extract(r"_ssp245_(PrecipData|TMaxData|TMinData)_Alwar\.csv$")[0]
    .value_counts()
    .to_string()
)

print("\nFile audit:")
display(audit)

print(
    "\nPASS: 39 files found and readable:",
    len(audit) == 39
    and audit["date_columns_present"].all()
    and (audit["rows"] > 0).all()
)


**[orig #171]** Leap-day audit  
<sub>✅ ran OK</sub>

In [ ]:

import os
import glob
import pandas as pd

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_daily_extracted"
)

files = sorted(glob.glob(os.path.join(OUT_DIR, "*_ssp245_*_Alwar.csv")))

date_summary = []
leap_summary = []

for path in files:
    name = os.path.basename(path)
    df = pd.read_csv(path, usecols=["Year", "Month", "Day"])

    dates = pd.to_datetime(
        {
            "year": pd.to_numeric(df["Year"], errors="coerce"),
            "month": pd.to_numeric(df["Month"], errors="coerce"),
            "day": pd.to_numeric(df["Day"], errors="coerce")
        },
        errors="coerce"
    )

    valid = dates.dropna()
    unique_dates = pd.DatetimeIndex(valid.unique()).sort_values()

    if len(unique_dates):
        expected = pd.date_range(
            unique_dates.min(), unique_dates.max(), freq="D"
        )
        missing = expected.difference(unique_dates)
        start, end = unique_dates.min(), unique_dates.max()
    else:
        missing = pd.DatetimeIndex([])
        start = end = pd.NaT

    date_summary.append({
        "file": name,
        "rows": len(df),
        "start": start,
        "end": end,
        "invalid_dates": int(dates.isna().sum()),
        "duplicate_dates": int(valid.duplicated().sum()),
        "missing_calendar_days": len(missing),
        "missing_dates": ", ".join(str(d.date()) for d in missing[:10])
    })

    if len(unique_dates):
        present = set(unique_dates)
        for year in range(start.year, end.year + 1):
            if not pd.Timestamp(year, 1, 1) in present:
                continue
            if not pd.Timestamp(year, 12, 31) in present:
                continue
            if pd.Timestamp(year, 2, 1).is_leap_year:
                leap_summary.append({
                    "file": name,
                    "year": year,
                    "Feb28": pd.Timestamp(year, 2, 28) in present,
                    "Feb29": pd.Timestamp(year, 2, 29) in present,
                    "Mar01": pd.Timestamp(year, 3, 1) in present
                })

date_audit = pd.DataFrame(date_summary)
leap_audit = pd.DataFrame(leap_summary)

date_audit.to_csv(
    os.path.join(OUT_DIR, "date_validation_audit.csv"), index=False
)
leap_audit.to_csv(
    os.path.join(OUT_DIR, "leap_day_audit.csv"), index=False
)

print("Files checked:", len(date_audit))
print("\nDate-quality summary:")
print(date_audit[
    ["invalid_dates", "duplicate_dates", "missing_calendar_days"]
].value_counts().to_string())

print("\nLeap-day presence:")
print(
    leap_audit.groupby(["Feb28", "Feb29", "Mar01"])
    .size().to_string()
)

print("\nFiles with missing calendar days:")
print(
    date_audit.loc[
        date_audit["missing_calendar_days"] > 0,
        ["file", "missing_calendar_days", "missing_dates"]
    ].to_string(index=False)
)

print("\nAudit reports saved in:", OUT_DIR)


**[orig #172]** Check the output folder  
<sub>✅ ran OK</sub>

In [ ]:

import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

OUT_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_daily_extracted"
)

print("Output folder exists:", os.path.isdir(OUT_DIR))

if os.path.isdir(OUT_DIR):
    print("CSV files found:", len([
        f for f in os.listdir(OUT_DIR)
        if f.endswith(".csv") and "_ssp245_" in f
    ]))
else:
    print("Parent folders:")
    p = "/content/drive/MyDrive/gw_framework/climate_forcing/Vimal_Mishra_CMIP6"
    print(os.path.exists(p), p)
    print("Please check the folder listing above.")


**[orig #173]** Insert missing 29 Feb rows (interpolate only those)  
<sub>✅ ran OK</sub>

In [ ]:

import os
import glob
import numpy as np
import pandas as pd

SOURCE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_daily_extracted"
)

OUTPUT_DIR = os.path.join(
    os.path.dirname(SOURCE_DIR),
    "Alwar_daily_leapday_corrected"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(glob.glob(
    os.path.join(SOURCE_DIR, "*_ssp245_*_Alwar.csv")
))

if len(files) != 39:
    raise RuntimeError(f"Expected 39 files; found {len(files)}")

file_log = []
value_log = []

for path in files:
    filename = os.path.basename(path)
    df = pd.read_csv(path)

    required = {"Year", "Month", "Day"}
    if not required.issubset(df.columns):
        raise ValueError(f"Date columns missing in {filename}")

    # Build and validate dates
    dates = pd.to_datetime(
        {
            "year": pd.to_numeric(df["Year"], errors="coerce"),
            "month": pd.to_numeric(df["Month"], errors="coerce"),
            "day": pd.to_numeric(df["Day"], errors="coerce")
        },
        errors="coerce"
    )

    if dates.isna().any():
        raise ValueError(f"Invalid dates in {filename}")

    if dates.duplicated().any():
        raise ValueError(f"Duplicate dates in {filename}")

    df = df.copy()
    df["_date"] = dates
    df = df.sort_values("_date").set_index("_date")

    original_dates = df.index
    full_index = pd.date_range(
        original_dates.min(), original_dates.max(), freq="D"
    )

    # Insert missing calendar dates, without filling unrelated gaps
    missing_dates = full_index.difference(original_dates)
    missing_non_leap = [
        d for d in missing_dates
        if not (d.month == 2 and d.day == 29)
    ]

    if missing_non_leap:
        raise ValueError(
            f"Unexpected missing dates in {filename}: "
            f"{missing_non_leap[:5]}"
        )

    df = df.reindex(full_index)
    grid_cols = [
        c for c in df.columns
        if c not in ["Year", "Month", "Day"]
    ]

    inserted_count = 0

    for date in missing_dates:
        prev_date = date - pd.Timedelta(days=1)
        next_date = date + pd.Timedelta(days=1)

        if prev_date not in df.index or next_date not in df.index:
            raise ValueError(
                f"Missing neighbours for {date.date()} in {filename}"
            )

        # Interpolate only the inserted 29 February row
        for col in grid_cols:
            before = pd.to_numeric(
                pd.Series([df.at[prev_date, col]]), errors="coerce"
            ).iloc[0]
            after = pd.to_numeric(
                pd.Series([df.at[next_date, col]]), errors="coerce"
            ).iloc[0]

            if pd.notna(before) and pd.notna(after):
                value = (before + after) / 2.0
                df.at[date, col] = value
            else:
                value = np.nan
                df.at[date, col] = np.nan

            value_log.append({
                "file": filename,
                "date": date.date(),
                "grid_column": col,
                "value_before_28Feb": before,
                "value_after_01Mar": after,
                "interpolated_value": value,
                "method": "Arithmetic midpoint",
                "valid_interpolation": pd.notna(value)
            })

        inserted_count += 1

    # Rebuild date columns; keep original column order
    df["Year"] = df.index.year
    df["Month"] = df.index.month
    df["Day"] = df.index.day

    df = df[["Year", "Month", "Day"] + grid_cols]
    df.to_csv(
        os.path.join(OUTPUT_DIR, filename),
        index=False
    )

    file_log.append({
        "file": filename,
        "original_rows": len(original_dates),
        "inserted_leap_days": inserted_count,
        "corrected_rows": len(df),
        "missing_non_leap_dates": len(missing_non_leap)
    })

# Save logs separately
file_log_df = pd.DataFrame(file_log)
value_log_df = pd.DataFrame(value_log)

file_log_df.to_csv(
    os.path.join(OUTPUT_DIR, "leapday_correction_file_log.csv"),
    index=False
)
value_log_df.to_csv(
    os.path.join(OUTPUT_DIR, "leapday_interpolation_value_log.csv"),
    index=False
)

print("Correction complete.")
print("Files processed:", len(file_log_df))
print("Files with inserted dates:",
      int((file_log_df["inserted_leap_days"] > 0).sum()))
print("Total inserted dates across files:",
      int(file_log_df["inserted_leap_days"].sum()))
print("Corrected data folder:", OUTPUT_DIR)
print("\nCorrection summary:")
print(file_log_df["corrected_rows"].value_counts().sort_index())
print("\nInterpolation values successfully calculated:",
      int(value_log_df["valid_interpolation"].sum()),
      "of", len(value_log_df))


**[orig #174]** Spatial validation step 1 — grid cells vs 14 blocks  
<sub>✅ ran OK</sub>

In [ ]:

# ALWAR CMIP6 SPATIAL VALIDATION — STEP 1

import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

GRID_CSV = os.path.join(
    BASE, "Alwar_spatial_selection",
    "Alwar_selected_grid_cells_inside_blocks.csv"
)

LGD_PATH = (
    "/content/drive/MyDrive/gw_framework/Shapefile/"
    "LGD_Blocks.geojson"
)

OUT = os.path.join(BASE, "Spatial_validation")
os.makedirs(OUT, exist_ok=True)

# Expected 14 study blocks
block_codes = list(range(574, 588))

# 1. Load the block polygons
blocks_all = gpd.read_file(LGD_PATH)

if "block_lgd" not in blocks_all.columns:
    raise ValueError(
        f"'block_lgd' not found. Columns: {blocks_all.columns.tolist()}"
    )

blocks_all["block_lgd"] = pd.to_numeric(
    blocks_all["block_lgd"], errors="coerce"
)

blocks = blocks_all[
    blocks_all["block_lgd"].isin(block_codes)
].copy()

if blocks["block_lgd"].nunique() != 14:
    raise ValueError(
        "Expected 14 unique block LGD codes; found "
        f"{blocks['block_lgd'].nunique()}. Check the boundary file."
    )

blocks = blocks.to_crs("EPSG:4326")
blocks = blocks.dissolve(
    by="block_lgd", as_index=False,
    aggfunc={"block_name": "first"}
)

# 2. Load the 12 selected cells
grid = pd.read_csv(GRID_CSV)

print("Grid CSV columns:", grid.columns.tolist())
print("Grid coordinate records:", len(grid))

# Identify longitude and latitude fields safely
lon_candidates = [
    c for c in grid.columns
    if c.lower() in ["longitude", "lon", "long", "x"]
]
lat_candidates = [
    c for c in grid.columns
    if c.lower() in ["latitude", "lat", "y"]
]

# Fall back to common names if present
if not lon_candidates and "longitude" in grid.columns:
    lon_candidates = ["longitude"]
if not lat_candidates and "latitude" in grid.columns:
    lat_candidates = ["latitude"]

if not lon_candidates or not lat_candidates:
    raise ValueError(
        "Could not identify longitude/latitude columns. "
        "Use the printed column names to set LON_COL and LAT_COL."
    )

LON_COL = lon_candidates[0]
LAT_COL = lat_candidates[0]

grid[LON_COL] = pd.to_numeric(grid[LON_COL], errors="coerce")
grid[LAT_COL] = pd.to_numeric(grid[LAT_COL], errors="coerce")
grid = grid.dropna(subset=[LON_COL, LAT_COL]).copy()

# The selection CSV contains repeated coordinates across models/variables.
# Reduce to unique coordinate pairs.
unique_grid = grid[[LON_COL, LAT_COL]].drop_duplicates().reset_index(drop=True)
unique_grid["grid_id"] = [
    f"ALW_{i:02d}" for i in range(1, len(unique_grid) + 1)
]

points = gpd.GeoDataFrame(
    unique_grid.copy(),
    geometry=gpd.points_from_xy(
        unique_grid[LON_COL], unique_grid[LAT_COL]
    ),
    crs="EPSG:4326"
)

print("\nUnique grid centres:", len(points))
print("Study blocks:", len(blocks))

# 3. Test centre-in-polygon membership
membership_rows = []

for _, b in blocks.iterrows():
    for _, p in points.iterrows():
        membership_rows.append({
            "block_lgd": int(b["block_lgd"]),
            "block_name": b["block_name"],
            "grid_id": p["grid_id"],
            "longitude": p[LON_COL],
            "latitude": p[LAT_COL],
            "centre_inside_block": bool(
                b.geometry.covers(p.geometry)
            )
        })

membership = pd.DataFrame(membership_rows)

centre_summary = (
    membership.groupby(
        ["block_lgd", "block_name"], as_index=False
    )
    .agg(
        grid_centres_inside=("centre_inside_block", "sum")
    )
)

centre_summary["centre_coverage"] = np.where(
    centre_summary["grid_centres_inside"] > 0,
    "At least one centre",
    "No centre inside block"
)

# 4. Approximate footprint overlap:
# A 0.25-degree box is only a preliminary approximation.
# It is NOT an equal-area square; use projected geometry for area calculations.
footprint_rows = []

for _, p in points.iterrows():
    half = 0.125
    footprint = box(
        p[LON_COL] - half,
        p[LAT_COL] - half,
        p[LON_COL] + half,
        p[LAT_COL] + half
    )

    for _, b in blocks.iterrows():
        overlap = footprint.intersection(b.geometry)

        footprint_rows.append({
            "block_lgd": int(b["block_lgd"]),
            "block_name": b["block_name"],
            "grid_id": p["grid_id"],
            "footprint_intersects_block": not overlap.is_empty,
            "overlap_area_degrees2": (
                overlap.area if not overlap.is_empty else 0.0
            )
        })

footprints = pd.DataFrame(footprint_rows)

footprint_summary = (
    footprints.groupby(
        ["block_lgd", "block_name"], as_index=False
    )
    .agg(
        approximate_cells_overlapping=(
            "footprint_intersects_block", "sum"
        )
    )
)

summary = centre_summary.merge(
    footprint_summary,
    on=["block_lgd", "block_name"],
    how="left"
)

summary.to_csv(
    os.path.join(OUT, "Alwar_block_grid_coverage_summary.csv"),
    index=False
)
membership.to_csv(
    os.path.join(OUT, "Alwar_block_grid_centre_membership.csv"),
    index=False
)
footprints.to_csv(
    os.path.join(OUT, "Alwar_approximate_footprint_intersections.csv"),
    index=False
)
points.drop(columns="geometry").to_csv(
    os.path.join(OUT, "Alwar_unique_grid_centres.csv"),
    index=False
)

print("\nBLOCK COVERAGE SUMMARY")
print(summary.to_string(index=False))

print("\nUnique cells:", len(points))
print("Blocks with centre coverage:",
      int((summary["grid_centres_inside"] > 0).sum()), "/ 14")
print("Blocks with approximate footprint overlap:",
      int((summary["approximate_cells_overlapping"] > 0).sum()), "/ 14")
print("\nReports saved to:", OUT)


**[orig #179]** Unique grid-cell centres  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

GRID_CSV = os.path.join(
    BASE,
    "Alwar_spatial_selection",
    "Alwar_selected_grid_cells_inside_blocks.csv"
)

print("File exists:", os.path.isfile(GRID_CSV))

grid = pd.read_csv(GRID_CSV)

cells = (
    grid[["longitude", "latitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)

cells.insert(
    0, "grid_id",
    [f"ALW_{i:02d}" for i in range(1, len(cells) + 1)]
)

print("\nUnique grid-cell centres:", len(cells))
print("\nGrid-cell coordinates:")
print(cells.to_string(index=False))

print("\nUnique longitude coordinates:")
print(np.sort(cells["longitude"].unique()))

print("\nUnique latitude coordinates:")
print(np.sort(cells["latitude"].unique()))

OUT = os.path.join(
    BASE, "Spatial_validation",
    "Alwar_unique_grid_centres_review.csv"
)
os.makedirs(os.path.dirname(OUT), exist_ok=True)
cells.to_csv(OUT, index=False)

print("\nSaved review file:", OUT)


**[orig #180]** Coordinate inventory; deduplicate cell locations  
<sub>✅ ran OK</sub>

In [ ]:

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

COORD_CSV = os.path.join(BASE, "ssp245_grid_coordinates.csv")
BLOCKS_FILE = (
    "/content/drive/MyDrive/gw_framework/Shapefile/LGD_Blocks.geojson"
)

OUT_DIR = os.path.join(BASE, "Spatial_validation")
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Check inputs
for path in [COORD_CSV, BLOCKS_FILE]:
    print(path, "exists:", os.path.isfile(path))

if not os.path.isfile(COORD_CSV):
    raise FileNotFoundError("Full coordinate inventory not found.")
if not os.path.isfile(BLOCKS_FILE):
    raise FileNotFoundError("LGD block GeoJSON not found.")

# 2. Read the complete coordinate inventory
coords = pd.read_csv(COORD_CSV)
print("\nCoordinate inventory columns:", coords.columns.tolist())
print("Coordinate inventory rows:", len(coords))

required = {"longitude", "latitude"}
if not required.issubset(coords.columns):
    raise ValueError(
        "Expected longitude and latitude columns. "
        f"Found: {coords.columns.tolist()}"
    )

coords["longitude"] = pd.to_numeric(coords["longitude"], errors="coerce")
coords["latitude"] = pd.to_numeric(coords["latitude"], errors="coerce")
coords = coords.dropna(subset=["longitude", "latitude"])

# Deduplicate spatial locations, retaining one representative coordinate
centres = (
    coords[["longitude", "latitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
centres["grid_id"] = [
    f"CELL_{i:05d}" for i in range(1, len(centres) + 1)
]

print("Unique grid centres in full inventory:", len(centres))

# 3. Read the 14 Alwar blocks
blocks = gpd.read_file(BLOCKS_FILE)

print("\nBlock file columns:", blocks.columns.tolist())

blocks["block_lgd"] = pd.to_numeric(
    blocks["block_lgd"], errors="coerce"
)
target_codes = list(range(574, 588))
alwar = blocks[blocks["block_lgd"].isin(target_codes)].copy()

if alwar.crs is None:
    raise ValueError("Block GeoJSON has no CRS; verify its coordinate system.")

alwar = alwar.to_crs("EPSG:4326")
alwar = alwar[alwar.geometry.notna() & ~alwar.geometry.is_empty].copy()

# Dissolve duplicate features per LGD code, if any
alwar = alwar[["block_lgd", "block_name", "geometry"]].dissolve(
    by=["block_lgd", "block_name"], as_index=False
)

found_codes = set(alwar["block_lgd"].astype(int))
missing_codes = sorted(set(target_codes) - found_codes)

print("Alwar blocks found:", len(alwar))
print("Missing LGD codes:", missing_codes)

if missing_codes:
    raise ValueError(f"Some target blocks were not found: {missing_codes}")

# 4. Create nominal 0.25-degree cells around each centre.
# This assumes centres are on a regular 0.25-degree grid.
half = 0.125
centres_gdf = gpd.GeoDataFrame(
    centres,
    geometry=[
        box(lon-half, lat-half, lon+half, lat+half)
        for lon, lat in zip(centres.longitude, centres.latitude)
    ],
    crs="EPSG:4326"
)

# 5. Project to UTM 43N before calculating intersection areas
metric_crs = "EPSG:32643"
cells_m = centres_gdf.to_crs(metric_crs)
blocks_m = alwar.to_crs(metric_crs)

# Restrict processing to cells whose footprints touch the study area
study_union = blocks_m.geometry.union_all()
candidate = cells_m[
    cells_m.geometry.intersects(study_union)
].copy()

print("\nCandidate cells touching the study area:", len(candidate))

# 6. Calculate intersection area and within-block area weight
records = []

for _, block in blocks_m.iterrows():
    block_geom = block.geometry
    block_area = block_geom.area

    for _, cell in candidate.iterrows():
        intersection = block_geom.intersection(cell.geometry)

        if not intersection.is_empty and intersection.area > 0:
            area = intersection.area
            records.append({
                "block_lgd": int(block["block_lgd"]),
                "block_name": block["block_name"],
                "grid_id": cell["grid_id"],
                "longitude": cell["longitude"],
                "latitude": cell["latitude"],
                "intersection_area_km2": area / 1e6,
                "block_area_km2": block_area / 1e6,
                "block_area_weight": area / block_area
            })

intersections = pd.DataFrame(records)

if intersections.empty:
    raise ValueError(
        "No intersections found. Check the coordinate inventory "
        "and the CRS of the block polygons."
    )

# Save detailed block-to-cell intersections
intersection_path = os.path.join(
    OUT_DIR, "Alwar_block_grid_intersections_nominal025.csv"
)
intersections.to_csv(intersection_path, index=False)

# Summarize coverage for each block
summary = (
    intersections.groupby(["block_lgd", "block_name"], as_index=False)
    .agg(
        intersecting_cells=("grid_id", "nunique"),
        intersected_area_km2=("intersection_area_km2", "sum"),
        sum_of_area_weights=("block_area_weight", "sum")
    )
)

block_areas = blocks_m[["block_lgd", "block_name", "geometry"]].copy()
block_areas["block_area_km2"] = block_areas.geometry.area / 1e6
block_areas = block_areas.drop(columns="geometry")

summary = summary.merge(
    block_areas, on=["block_lgd", "block_name"], how="right"
)
summary["intersecting_cells"] = (
    summary["intersecting_cells"].fillna(0).astype(int)
)
summary["intersected_area_km2"] = (
    summary["intersected_area_km2"].fillna(0)
)
summary["sum_of_area_weights"] = (
    summary["sum_of_area_weights"].fillna(0)
)

summary["coverage_check"] = np.where(
    np.isclose(summary["sum_of_area_weights"], 1.0, atol=0.02),
    "Approximately complete",
    "Check geometry / grid assumptions"
)

summary_path = os.path.join(
    OUT_DIR, "Alwar_block_grid_coverage_nominal025.csv"
)
summary.to_csv(summary_path, index=False)

print("\nBLOCK COVERAGE SUMMARY")
print(summary.to_string(index=False))

print("\nDistinct intersecting cells:", intersections["grid_id"].nunique())
print("Intersection table:", intersection_path)
print("Coverage summary:", summary_path)


**[orig #181]** **MAIN** — 22 intersecting cells and block–cell area weights  
<sub>✅ ran OK</sub>

In [ ]:

import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

COORD_CSV = os.path.join(BASE, "ssp245_grid_coordinates.csv")
INTERSECTION_CSV = os.path.join(
    BASE, "Spatial_validation",
    "Alwar_block_grid_intersections_nominal025.csv"
)

OUT_DIR = os.path.join(BASE, "Spatial_validation")

coords = pd.read_csv(COORD_CSV)
intersections = pd.read_csv(INTERSECTION_CSV)

# Keep the 22 distinct spatial cells
cells = (
    intersections[["grid_id", "longitude", "latitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Identify the matching source columns in each model-variable file
mapping = coords.merge(
    cells[["longitude", "latitude", "grid_id"]],
    on=["longitude", "latitude"],
    how="inner",
    validate="many_to_one"
)

# Check each source file has exactly one mapping per intersecting cell
per_file = mapping.groupby("source_file").agg(
    mapped_cells=("grid_id", "nunique"),
    mapped_rows=("grid_id", "size")
).reset_index()

print("Distinct intersecting cells:", cells["grid_id"].nunique())
print("Source files:", mapping["source_file"].nunique())
print("\nCell mappings per source file:")
print(per_file.to_string(index=False))

expected = 22
bad = per_file[per_file["mapped_cells"] != expected]

if len(bad):
    print("\nWARNING: Some files do not map all 22 cells:")
    print(bad.to_string(index=False))
else:
    print("\nPASS: Every source file maps all 22 intersecting cells.")

# Save full mapping for reproducibility
mapping_path = os.path.join(
    OUT_DIR, "Alwar_22cell_source_column_mapping.csv"
)
mapping.to_csv(mapping_path, index=False)

print("\nMapping saved:", mapping_path)
print("\nExample mappings:")
print(
    mapping[
        ["model", "variable", "source_file", "grid_id",
         "longitude", "latitude", "column_index_zero_based"]
    ].head(15).to_string(index=False)
)


**[orig #183]** Raw lines of a source file (delimiter check)  
<sub>✅ ran OK</sub>

In [ ]:

source = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India/"
    "ACCESS-CM2/ssp245/PrecipData"
)

# Display the first 3 raw lines exactly as stored
with open(source, "r", encoding="utf-8-sig") as f:
    for i in range(3):
        line = f.readline()
        print(f"\n--- RAW LINE {i+1} ---")
        print(repr(line[:1000]))
        print("Comma count:", line.count(","))
        print("Tab count:", line.count("\t"))
        print("Semicolon count:", line.count(";"))

# Compare common delimiters
import pandas as pd

for delimiter_name, delimiter in [
    ("comma", ","),
    ("tab", "\t"),
    ("semicolon", ";"),
    ("whitespace", r"\s+")
]:
    try:
        df = pd.read_csv(
            source, header=None, nrows=2,
            sep=delimiter, engine="python"
        )
        print(f"\n{delimiter_name}: {df.shape[1]} columns")
        print(df.iloc[:, :6].to_string(index=False, header=False))
    except Exception as e:
        print(f"\n{delimiter_name}: ERROR - {e}")


**[orig #184]** Raw lines and file size  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path

p = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India/"
    "ACCESS-CM2/ssp245/PrecipData"
)

print("File exists:", p.exists())

if p.exists():
    print("File size (GB):", round(p.stat().st_size / 1e9, 3))

    with open(p, "r", encoding="utf-8-sig", errors="replace") as f:
        for i in range(3):
            line = f.readline()
            print(f"\nRAW LINE {i+1} (first 300 chars):")
            print(repr(line[:300]))

    with open(p, "r", encoding="utf-8-sig", errors="replace") as f:
        first_line = f.readline()

    print("\nSEPARATOR COUNTS IN FIRST LINE:")
    for sep, name in [
        (",", "comma"),
        ("\t", "tab"),
        (";", "semicolon"),
        ("|", "pipe"),
    ]:
        print(f"{name}: {first_line.count(sep)}")
else:
    print("Check the file path and mounted Google Drive.")

**[orig #185]** Parse first rows with whitespace separator  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
from pathlib import Path

source = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India/"
    "ACCESS-CM2/ssp245/PrecipData"
)

# Read only the first three rows using whitespace separation
sample = pd.read_csv(
    source,
    sep=r"\s+",
    header=None,
    nrows=3,
    engine="c"
)

print("Parsed shape:", sample.shape)
print("\nFirst 8 columns of each row:")
print(sample.iloc[:, :8].to_string(index=False, header=False))

print("\nCoordinate columns available:", sample.shape[1] - 3)
print("First longitude:", sample.iloc[0, 3])
print("First latitude:", sample.iloc[1, 3])

# Check selected grid-column indices from the earlier workflow
print("\nTest previously mapped grid indices:")
for idx in [1449, 1500, 1800, 1890]:
    if idx < sample.shape[1]:
        print(
            f"Column {idx}: "
            f"longitude={sample.iloc[0, idx]}, "
            f"latitude={sample.iloc[1, idx]}, "
            f"first daily value={sample.iloc[2, idx]}"
        )
    else:
        print(f"Column {idx}: outside parsed file width")

**[orig #186]** Check selected grid indices  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

source = base / "India/ACCESS-CM2/ssp245/PrecipData"

selection_file = (
    base / "Alwar_grid_analysis/"
    "Alwar_spatial_selection/"
    "Alwar_selected_grid_cells_inside_blocks.csv"
)

# Read the selected cells
selected = pd.read_csv(selection_file)

print("Selection file columns:", selected.columns.tolist())
print("Selection records:", len(selected))
print("Unique grid indices:", selected["column_index_zero_based"].nunique())

# Read only the two coordinate rows from the original source
coords = pd.read_csv(
    source,
    sep=r"\s+",
    header=None,
    nrows=2,
    engine="c"
)

results = []

for _, r in selected.drop_duplicates(
    subset=["column_index_zero_based"]
).iterrows():
    idx = int(r["column_index_zero_based"])

    if idx >= coords.shape[1] or idx < 3:
        results.append({
            "column_index": idx,
            "selected_lon": r["longitude"],
            "selected_lat": r["latitude"],
            "source_lon": np.nan,
            "source_lat": np.nan,
            "match": False,
            "reason": "Index outside valid grid columns"
        })
        continue

    lon = float(coords.iloc[0, idx])
    lat = float(coords.iloc[1, idx])

    match = (
        np.isclose(lon, float(r["longitude"])) and
        np.isclose(lat, float(r["latitude"]))
    )

    results.append({
        "column_index": idx,
        "selected_lon": r["longitude"],
        "selected_lat": r["latitude"],
        "source_lon": lon,
        "source_lat": lat,
        "match": match,
        "reason": "OK" if match else "Coordinate mismatch"
    })

check = pd.DataFrame(results)

print("\nCoordinate validation:")
print(check.to_string(index=False))
print("\nAll coordinates match:", check["match"].all())

**[orig #187]** Candidate grid cells near Alwar; grid spacing  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from shapely.geometry import box

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

source = base / "India/ACCESS-CM2/ssp245/PrecipData"

blocks_path = Path(
    "/content/drive/MyDrive/gw_framework/Shapefile/LGD_Blocks.geojson"
)

outdir = base / "Alwar_grid_analysis/Area_weighted_spatial_selection"
outdir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Read grid coordinates
# --------------------------------------------------
coords = pd.read_csv(
    source, sep=r"\s+", header=None, nrows=2, engine="c"
)

grid = pd.DataFrame({
    "column_index_zero_based": np.arange(3, coords.shape[1]),
    "longitude": pd.to_numeric(coords.iloc[0, 3:], errors="coerce").to_numpy(),
    "latitude": pd.to_numeric(coords.iloc[1, 3:], errors="coerce").to_numpy()
}).dropna(subset=["longitude", "latitude"])

grid = grid[
    grid["longitude"].between(76.0, 77.4) &
    grid["latitude"].between(26.9, 28.4)
].copy()

print("Candidate grid cells near Alwar:", len(grid))
print("Unique coordinate pairs:", grid[["longitude", "latitude"]].drop_duplicates().shape[0])

# Check spacing between unique coordinates
lons = np.sort(grid["longitude"].unique())
lats = np.sort(grid["latitude"].unique())

print("Longitude spacing values:", np.unique(np.round(np.diff(lons), 6)))
print("Latitude spacing values:", np.unique(np.round(np.diff(lats), 6)))

# --------------------------------------------------
# 3. Read the 14 Alwar blocks
# --------------------------------------------------
blocks = gpd.read_file(blocks_path)

alwar = blocks[
    blocks["block_lgd"].between(574, 587)
].copy()

if len(alwar) != 14:
    raise ValueError(
        f"Expected 14 Alwar blocks; found {len(alwar)}. "
        "Check the LGD codes and block layer."
    )

if alwar.crs is None:
    raise ValueError("Block polygons have no defined CRS.")

alwar = alwar.to_crs("EPSG:4326")
alwar = alwar[["block_lgd", "block_name", "geometry"]].copy()
alwar["geometry"] = alwar.geometry.make_valid()

# --------------------------------------------------
# 4. Build approximate 0.25-degree grid-cell footprints
# --------------------------------------------------
# This assumes the source grid centres are spaced by 0.25 degrees.
# The spacing diagnostics above must be checked before interpreting results.

cell_size = 0.25
half = cell_size / 2

grid["geometry"] = [
    box(lon - half, lat - half, lon + half, lat + half)
    for lon, lat in zip(grid["longitude"], grid["latitude"])
]

cells = gpd.GeoDataFrame(grid, geometry="geometry", crs="EPSG:4326")

# Project both layers before calculating areas.
# EPSG:32643 is UTM Zone 43N, suitable for this region.
cells_m = cells.to_crs("EPSG:32643")
blocks_m = alwar.to_crs("EPSG:32643")

# --------------------------------------------------
# 5. Intersect cells with blocks
# --------------------------------------------------
intersections = gpd.overlay(
    blocks_m,
    cells_m,
    how="intersection",
    keep_geom_type=True
)

intersections["intersection_area_m2"] = intersections.geometry.area

# Remove zero/tiny sliver intersections
intersections = intersections[
    intersections["intersection_area_m2"] > 1
].copy()

# Fraction of each block covered by each intersecting cell
block_area = blocks_m.set_index("block_lgd").geometry.area
intersections["block_area_m2"] = intersections["block_lgd"].map(block_area)

intersections["block_area_fraction"] = (
    intersections["intersection_area_m2"] /
    intersections["block_area_m2"]
)

# Normalized weights for calculating a block-level climate average
intersections["weight_within_intersected_area"] = (
    intersections["intersection_area_m2"] /
    intersections.groupby("block_lgd")["intersection_area_m2"].transform("sum")
)

# --------------------------------------------------
# 6. Save results
# --------------------------------------------------
cols = [
    "block_lgd", "block_name",
    "column_index_zero_based", "longitude", "latitude",
    "intersection_area_m2", "block_area_m2",
    "block_area_fraction", "weight_within_intersected_area"
]

result = pd.DataFrame(intersections[cols]).sort_values(
    ["block_lgd", "column_index_zero_based"]
)

result.to_csv(outdir / "Alwar_block_grid_intersection_weights.csv", index=False)

cell_summary = (
    result.groupby(["column_index_zero_based", "longitude", "latitude"])
    .agg(
        blocks_intersected=("block_lgd", "nunique"),
        total_intersection_area_m2=("intersection_area_m2", "sum")
    )
    .reset_index()
)

cell_summary.to_csv(outdir / "Alwar_intersecting_unique_grid_cells.csv", index=False)

coverage = (
    result.groupby(["block_lgd", "block_name"])
    .agg(
        intersecting_cells=("column_index_zero_based", "nunique"),
        summed_block_area_fraction=("block_area_fraction", "sum")
    )
    .reset_index()
)

coverage = alwar[["block_lgd", "block_name"]].merge(
    coverage, on=["block_lgd", "block_name"], how="left"
)

coverage["intersecting_cells"] = coverage["intersecting_cells"].fillna(0).astype(int)
coverage.to_csv(outdir / "Alwar_block_spatial_coverage_summary.csv", index=False)

print("\nUnique intersecting cells:", len(cell_summary))
print("\nCoverage by block:")
print(coverage.to_string(index=False))

print("\nSaved outputs to:", outdir)

## 4. Block-weighted daily forcing

**[orig #189]** Check inputs/outputs for block-weighted extraction  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import os

root = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

weights_file = (
    root / "Alwar_grid_analysis/Area_weighted_spatial_selection/"
    "Alwar_block_grid_intersection_weights.csv"
)

output_dir = (
    root / "Alwar_grid_analysis/Alwar_block_daily_forcing"
)

print("Google Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Source directory exists:", (root / "India").exists())
print("Weights file exists:", weights_file.exists())
print("Output directory exists:", output_dir.exists())

if weights_file.exists():
    import pandas as pd
    w = pd.read_csv(weights_file)
    print("\nWeight rows:", len(w))
    print("Unique blocks:", w["block_lgd"].nunique())
    print("Unique grid indices:", w["column_index_zero_based"].nunique())

if (root / "India").exists():
    source_files = [
        p for p in (root / "India").glob("*/ssp245/*")
        if p.is_file() and p.name in ["PrecipData", "TMaxData", "TMinData"]
    ]
    print("\nSource files found:", len(source_files))
    print("First five source files:")
    for p in source_files[:5]:
        print(p, f"({p.stat().st_size / 1e9:.3f} GB)")

if output_dir.exists():
    print("\nOutput folder contents:")
    for p in list(output_dir.iterdir())[:20]:
        print(p.name, f"({p.stat().st_size / 1e6:.2f} MB)")

print("\nSearching for block-level outputs and audit logs...")
for pattern in ["*_AlwarBlocks.csv", "processing_audit_log.csv"]:
    matches = list(root.rglob(pattern))
    print(f"{pattern}: {len(matches)} found")
    for p in matches[:10]:
        print(" ", p)

**[orig #190]** Block-weighted daily values — test run  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import traceback

base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

source = base / "India/ACCESS-CM2/ssp245/PrecipData"

weights_file = (
    base / "Alwar_grid_analysis/Area_weighted_spatial_selection/"
    "Alwar_block_grid_intersection_weights.csv"
)

test_dir = base / "Alwar_grid_analysis/TEST_one_file"
test_dir.mkdir(parents=True, exist_ok=True)

try:
    print("1. Reading weights...")
    weights = pd.read_csv(weights_file)
    weights["column_index_zero_based"] = (
        weights["column_index_zero_based"].astype(int)
    )

    blocks = weights[["block_lgd", "block_name"]].drop_duplicates()
    grid_indices = sorted(
        weights["column_index_zero_based"].unique()
    )

    print("Blocks:", len(blocks))
    print("Selected grid cells:", len(grid_indices))

    # Only read the columns needed for this test.
    usecols = sorted(set([0, 1, 2] + grid_indices))

    print("2. Reading source file...")
    df = pd.read_csv(
        source,
        sep=r"\s+",
        header=None,
        usecols=usecols,
        engine="c",
        low_memory=False
    )
    df.columns = usecols
    print("Source rows:", len(df))
    print("Columns read:", len(df.columns))

    print("3. Constructing dates...")
    dates = pd.to_datetime({
        "year": pd.to_numeric(df[0], errors="coerce"),
        "month": pd.to_numeric(df[1], errors="coerce"),
        "day": pd.to_numeric(df[2], errors="coerce")
    }, errors="coerce")

    if dates.isna().any():
        raise ValueError("Invalid dates found in the source file.")

    climate = df[grid_indices].apply(pd.to_numeric, errors="coerce")
    climate.index = dates

    print("Date range:", climate.index.min(), "to", climate.index.max())
    print("Duplicate dates:", climate.index.duplicated().sum())

    print("4. Calculating block averages...")
    result = pd.DataFrame(index=climate.index)

    for _, block in blocks.iterrows():
        block_id = int(block["block_lgd"])
        block_name = block["block_name"]

        wb = weights[weights["block_lgd"] == block_id]
        cols = wb["column_index_zero_based"].tolist()
        w = wb.set_index("column_index_zero_based")[
            "weight_within_intersected_area"
        ].reindex(cols).to_numpy(dtype=float)

        values = climate[cols].to_numpy(dtype=float)
        valid = np.isfinite(values)
        denominator = (valid * w.reshape(1, -1)).sum(axis=1)
        numerator = np.nansum(values * w.reshape(1, -1), axis=1)

        result[f"{block_id}_{block_name}"] = np.divide(
            numerator,
            denominator,
            out=np.full(len(climate), np.nan),
            where=denominator > 0
        )

    print("5. Saving test output...")
    output = result.copy()
    output.insert(0, "Date", output.index.strftime("%Y-%m-%d"))
    output.insert(1, "Year", output.index.year)
    output.insert(2, "Month", output.index.month)
    output.insert(3, "Day", output.index.day)

    test_file = test_dir / "ACCESS-CM2_ssp245_PrecipData_TEST.csv"
    output.to_csv(test_file, index=False)

    print("\nTEST SUCCESSFUL")
    print("Output file:", test_file)
    print("Output shape:", output.shape)
    print("Missing values:", int(result.isna().sum().sum()))
    print("First five rows:")
    print(output.head().to_string(index=False))

except Exception:
    print("\nTEST FAILED — full error:")
    traceback.print_exc()

**[orig #192]** Date-field validation  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

source = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India/"
    "ACCESS-CM2/ssp245/PrecipData"
)

dates_df = pd.read_csv(
    source,
    sep=r"\s+",
    header=None,
    usecols=[0, 1, 2],
    engine="c"
)

dates_df.columns = ["Year", "Month", "Day"]

print("Total rows:", len(dates_df))
print("\nFirst 8 rows:")
print(dates_df.head(8).to_string(index=False))

print("\nLast 8 rows:")
print(dates_df.tail(8).to_string(index=False))

# Identify non-numeric date fields
for col in ["Year", "Month", "Day"]:
    dates_df[col] = pd.to_numeric(dates_df[col], errors="coerce")

numeric_mask = dates_df[["Year", "Month", "Day"]].notna().all(axis=1)

print("\nRows with non-numeric date fields:", (~numeric_mask).sum())
if (~numeric_mask).any():
    print(dates_df.loc[~numeric_mask].head(20).to_string())

# Identify calendar-invalid dates
valid_dates = pd.to_datetime(
    dates_df.rename(columns={
        "Year": "year",
        "Month": "month",
        "Day": "day"
    }),
    errors="coerce"
)

invalid_mask = valid_dates.isna()

print("\nInvalid calendar rows:", invalid_mask.sum())
if invalid_mask.any():
    print(dates_df.loc[invalid_mask].head(30).to_string())

# Check duplicate dates among valid rows
valid = valid_dates.dropna()
duplicates = valid[valid.duplicated(keep=False)]

print("\nRows participating in duplicate dates:", len(duplicates))
if len(duplicates):
    print(duplicates.head(30).to_string())

print("\nYear counts:")
print(dates_df["Year"].value_counts(dropna=False).sort_index().to_string())

**[orig #193]** Block-weighted test using weights and metadata rows  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

source = base / "India/ACCESS-CM2/ssp245/PrecipData"

weights_file = (
    base / "Alwar_grid_analysis/Area_weighted_spatial_selection/"
    "Alwar_block_grid_intersection_weights.csv"
)

test_dir = base / "Alwar_grid_analysis/TEST_one_file"
test_dir.mkdir(parents=True, exist_ok=True)

# Read spatial weights
weights = pd.read_csv(weights_file)
weights["column_index_zero_based"] = (
    weights["column_index_zero_based"].astype(int)
)

blocks = weights[["block_lgd", "block_name"]].drop_duplicates()
grid_indices = sorted(weights["column_index_zero_based"].unique())
usecols = sorted(set([0, 1, 2] + grid_indices))

# Read selected columns, including the two metadata rows
df = pd.read_csv(
    source,
    sep=r"\s+",
    header=None,
    usecols=usecols,
    engine="c",
    low_memory=False
)
df.columns = usecols

print("Rows including metadata:", len(df))

# IMPORTANT: the first two rows contain grid coordinates, not dates
df = df.iloc[2:].copy()

# Construct dates from daily records only
date_parts = df[[0, 1, 2]].apply(pd.to_numeric, errors="coerce")
dates = pd.to_datetime(
    date_parts.rename(
        columns={0: "year", 1: "month", 2: "day"}
    ),
    errors="coerce"
)

if dates.isna().any():
    raise ValueError("Invalid dates remain after skipping metadata rows.")

if dates.duplicated().any():
    raise ValueError("Duplicate dates found.")

expected = pd.date_range("2015-01-01", "2100-12-31", freq="D")
missing_dates = expected.difference(pd.DatetimeIndex(dates))
unexpected_dates = pd.DatetimeIndex(dates).difference(expected)

print("Daily rows:", len(df))
print("Date range:", dates.min(), "to", dates.max())
print("Missing calendar dates:", len(missing_dates))
print("Unexpected dates:", len(unexpected_dates))

if len(missing_dates) or len(unexpected_dates):
    raise ValueError("Calendar validation failed; inspect dates before proceeding.")

# Climate grid-cell values
climate = df[grid_indices].apply(pd.to_numeric, errors="coerce")
climate.index = pd.DatetimeIndex(dates)
climate.index.name = "Date"

# Calculate weighted daily values for each block
result = pd.DataFrame(index=climate.index)

for _, block in blocks.iterrows():
    block_id = int(block["block_lgd"])
    block_name = block["block_name"]

    wb = weights[weights["block_lgd"] == block_id]
    cols = wb["column_index_zero_based"].astype(int).tolist()
    w = wb.set_index("column_index_zero_based")[
        "weight_within_intersected_area"
    ].reindex(cols).to_numpy(dtype=float)

    values = climate[cols].to_numpy(dtype=float)
    valid = np.isfinite(values)

    numerator = np.nansum(values * w.reshape(1, -1), axis=1)
    denominator = (valid * w.reshape(1, -1)).sum(axis=1)

    result[f"{block_id}_{block_name}"] = np.divide(
        numerator,
        denominator,
        out=np.full(len(climate), np.nan),
        where=denominator > 0
    )

output = result.copy()
output.insert(0, "Date", output.index.strftime("%Y-%m-%d"))
output.insert(1, "Year", output.index.year)
output.insert(2, "Month", output.index.month)
output.insert(3, "Day", output.index.day)

test_file = test_dir / "ACCESS-CM2_ssp245_PrecipData_TEST.csv"
output.to_csv(test_file, index=False)

print("\nTEST SUCCESSFUL")
print("Saved:", test_file)
print("Output shape:", output.shape)
print("Missing block-day values:", int(result.isna().sum().sum()))
print(output.head().to_string(index=False))

**[orig #194]** **MAIN** — block-weighted daily forcing for 14 blocks × 13 GCMs × 3 variables  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import traceback

# =========================================================
# PATHS
# =========================================================
base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

source_root = base / "India"

weights_file = (
    base / "Alwar_grid_analysis/Area_weighted_spatial_selection/"
    "Alwar_block_grid_intersection_weights.csv"
)

outdir = base / "Alwar_grid_analysis/Alwar_block_daily_forcing"
outdir.mkdir(parents=True, exist_ok=True)

audit_file = outdir / "processing_audit_log.csv"
error_file = outdir / "processing_errors.csv"

expected_dates = pd.date_range("2015-01-01", "2100-12-31", freq="D")

# =========================================================
# LOAD AND VALIDATE WEIGHTS
# =========================================================
weights = pd.read_csv(weights_file)
weights["column_index_zero_based"] = (
    weights["column_index_zero_based"].astype(int)
)

blocks = (
    weights[["block_lgd", "block_name"]]
    .drop_duplicates()
    .sort_values("block_lgd")
)

grid_indices = sorted(
    weights["column_index_zero_based"].unique()
)

if len(blocks) != 14:
    raise ValueError(f"Expected 14 blocks, found {len(blocks)}")

weight_sums = weights.groupby("block_lgd")[
    "weight_within_intersected_area"
].sum()

if not np.allclose(weight_sums.values, 1.0):
    raise ValueError("Spatial weights do not sum to 1 for every block.")

# Discover source files
source_files = sorted(
    p for p in source_root.glob("*/ssp245/*")
    if p.is_file()
    and p.name in {"PrecipData", "TMaxData", "TMinData"}
)

if len(source_files) != 39:
    raise ValueError(f"Expected 39 source files, found {len(source_files)}")

print("Source files:", len(source_files))
print("Blocks:", len(blocks))
print("Grid cells:", len(grid_indices))
print("Output folder:", outdir)

# Load previous audit records for restart support
if audit_file.exists():
    previous = pd.read_csv(audit_file)
    audit_records = previous.to_dict("records")
else:
    audit_records = []

# Retain only the latest audit record per output filename
audit_by_file = {
    str(r["output_file"]): r
    for r in audit_records
    if "output_file" in r
}

errors = []

# =========================================================
# PROCESS ONE FILE AT A TIME
# =========================================================
for i, source in enumerate(source_files, start=1):
    model = source.parent.parent.name
    variable = source.name

    output_file = outdir / f"{model}_ssp245_{variable}_AlwarBlocks.csv"
    output_key = str(output_file)

    # Skip an existing output only if it passes basic validation
    if output_file.exists():
        try:
            existing = pd.read_csv(output_file, usecols=["Date"])
            existing_dates = pd.to_datetime(existing["Date"], errors="coerce")

            if (
                len(existing_dates) == len(expected_dates)
                and not existing_dates.isna().any()
                and existing_dates.is_monotonic_increasing
                and not existing_dates.duplicated().any()
                and existing_dates.iloc[0] == expected_dates[0]
                and existing_dates.iloc[-1] == expected_dates[-1]
            ):
                print(f"[{i}/39] SKIP validated output: {output_file.name}")
                continue
        except Exception:
            pass

    print(f"\n[{i}/39] Processing {model} | {variable}")

    try:
        usecols = sorted(set([0, 1, 2] + grid_indices))

        df = pd.read_csv(
            source,
            sep=r"\s+",
            header=None,
            usecols=usecols,
            engine="c",
            low_memory=False
        )
        df.columns = usecols

        # First two rows contain longitude/latitude metadata
        df = df.iloc[2:].copy()

        date_parts = df[[0, 1, 2]].apply(pd.to_numeric, errors="coerce")

        dates = pd.to_datetime(
            date_parts.rename(
                columns={0: "year", 1: "month", 2: "day"}
            ),
            errors="coerce"
        )

        if dates.isna().any():
            raise ValueError("Invalid dates remain after metadata removal.")

        if dates.duplicated().any():
            raise ValueError("Duplicate dates found.")

        dates = pd.DatetimeIndex(dates)

        if len(dates) != len(expected_dates):
            raise ValueError(
                f"Expected {len(expected_dates)} daily records, got {len(dates)}"
            )

        if not dates.equals(expected_dates):
            raise ValueError("Source dates do not exactly match expected calendar.")

        climate = df[grid_indices].apply(pd.to_numeric, errors="coerce")
        climate.index = dates
        climate.index.name = "Date"

        result = pd.DataFrame(index=dates)

        for _, block in blocks.iterrows():
            block_id = int(block["block_lgd"])
            block_name = block["block_name"]

            wb = weights[weights["block_lgd"] == block_id]
            cols = wb["column_index_zero_based"].astype(int).tolist()

            w = wb.set_index("column_index_zero_based")[
                "weight_within_intersected_area"
            ].reindex(cols).to_numpy(dtype=float)

            values = climate[cols].to_numpy(dtype=float)
            valid = np.isfinite(values)

            numerator = np.nansum(values * w.reshape(1, -1), axis=1)
            denominator = (valid * w.reshape(1, -1)).sum(axis=1)

            result[f"{block_id}_{block_name}"] = np.divide(
                numerator,
                denominator,
                out=np.full(len(climate), np.nan),
                where=denominator > 0
            )

        # Preserve variable-specific values without altering units
        output = result.copy()
        output.insert(0, "Day", output.index.day)
        output.insert(0, "Month", output.index.month)
        output.insert(0, "Year", output.index.year)
        output.insert(0, "Date", output.index.strftime("%Y-%m-%d"))

        # Write to a temporary file, then replace final output
        temporary_file = output_file.with_suffix(".tmp")
        output.to_csv(temporary_file, index=False)
        temporary_file.replace(output_file)

        record = {
            "model": model,
            "scenario": "ssp245",
            "variable": variable,
            "source_file": str(source),
            "output_file": str(output_file),
            "rows": len(output),
            "start_date": str(dates.min().date()),
            "end_date": str(dates.max().date()),
            "missing_block_day_values": int(result.isna().sum().sum()),
            "status": "SUCCESS"
        }

        audit_by_file[output_key] = record
        pd.DataFrame(audit_by_file.values()).to_csv(audit_file, index=False)

        print(
            f"  Saved {output_file.name} | "
            f"rows={len(output)} | "
            f"missing values={record['missing_block_day_values']}"
        )

    except Exception as exc:
        error_record = {
            "model": model,
            "variable": variable,
            "source_file": str(source),
            "error": repr(exc)
        }
        errors.append(error_record)
        pd.DataFrame(errors).to_csv(error_file, index=False)
        print("  FAILED:", repr(exc))
        traceback.print_exc()

# =========================================================
# FINAL SUMMARY
# =========================================================
print("\n========== FINAL SUMMARY ==========")
print("Output CSV files:", len(list(outdir.glob("*_AlwarBlocks.csv"))))
print("Audit log exists:", audit_file.exists())

if audit_file.exists():
    audit = pd.read_csv(audit_file)
    print("Successful audit records:", (audit["status"] == "SUCCESS").sum())
    print("\nAudit summary by variable:")
    print(audit.groupby(["variable", "status"]).size().to_string())

if errors:
    print("\nFiles with errors:", len(errors))
    print(pd.DataFrame(errors).to_string(index=False))
else:
    print("No processing errors recorded.")


**[orig #195]** Resume/repair the extraction (keeps finished files)  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import traceback

base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)
source_root = base / "India"

weights_file = (
    base / "Alwar_grid_analysis/Area_weighted_spatial_selection/"
    "Alwar_block_grid_intersection_weights.csv"
)
outdir = base / "Alwar_grid_analysis/Alwar_block_daily_forcing"
outdir.mkdir(parents=True, exist_ok=True)

audit_file = outdir / "processing_audit_log.csv"
error_file = outdir / "processing_errors.csv"

expected_dates = pd.date_range("2015-01-01", "2100-12-31", freq="D")

weights = pd.read_csv(weights_file)
weights["column_index_zero_based"] = (
    weights["column_index_zero_based"].astype(int)
)
blocks = (
    weights[["block_lgd", "block_name"]]
    .drop_duplicates()
    .sort_values("block_lgd")
)
grid_indices = sorted(weights["column_index_zero_based"].unique())

source_files = sorted(
    p for p in source_root.glob("*/ssp245/*")
    if p.is_file() and p.name in {"PrecipData", "TMaxData", "TMinData"}
)

if len(source_files) != 39:
    raise ValueError(f"Expected 39 source files, found {len(source_files)}")

def output_is_valid(path):
    if not path.exists():
        return False
    try:
        dates = pd.to_datetime(
            pd.read_csv(path, usecols=["Date"])["Date"],
            errors="coerce"
        )
        return (
            len(dates) == len(expected_dates)
            and not dates.isna().any()
            and not dates.duplicated().any()
            and dates.is_monotonic_increasing
            and dates.iloc[0] == expected_dates[0]
            and dates.iloc[-1] == expected_dates[-1]
        )
    except Exception:
        return False

# Preserve previous successful audit entries
audit_records = {}
if audit_file.exists():
    old = pd.read_csv(audit_file)
    for rec in old.to_dict("records"):
        if "output_file" in rec:
            audit_records[str(rec["output_file"])] = rec

errors = []

for i, source in enumerate(source_files, start=1):
    model = source.parent.parent.name
    variable = source.name
    output_file = outdir / f"{model}_ssp245_{variable}_AlwarBlocks.csv"

    if output_is_valid(output_file):
        print(f"[{i}/39] Already valid: {output_file.name}")
        continue

    print(f"\n[{i}/39] Processing: {model} | {variable}")

    try:
        usecols = sorted(set([0, 1, 2] + grid_indices))
        df = pd.read_csv(
            source,
            sep=r"\s+",
            header=None,
            usecols=usecols,
            engine="c",
            low_memory=False
        )
        df.columns = usecols

        # Skip the longitude and latitude metadata rows
        df = df.iloc[2:].copy()

        date_parts = df[[0, 1, 2]].apply(pd.to_numeric, errors="coerce")
        dates = pd.to_datetime(
            date_parts.rename(
                columns={0: "year", 1: "month", 2: "day"}
            ),
            errors="coerce"
        )

        if dates.isna().any() or dates.duplicated().any():
            raise ValueError("Invalid or duplicate source dates")

        dates = pd.DatetimeIndex(dates)
        missing_dates = expected_dates.difference(dates)
        unexpected_dates = dates.difference(expected_dates)

        if len(unexpected_dates):
            raise ValueError(f"Unexpected source dates: {list(unexpected_dates[:5])}")

        non_leap_missing = [
            d for d in missing_dates
            if not (d.month == 2 and d.day == 29)
        ]
        if non_leap_missing:
            raise ValueError(
                f"Missing non-Feb-29 dates: {non_leap_missing[:5]}"
            )

        # Calculate block-weighted values on the model's original dates
        climate = df[grid_indices].apply(pd.to_numeric, errors="coerce")
        climate.index = dates
        climate.index.name = "Date"

        result = pd.DataFrame(index=dates)

        for _, block in blocks.iterrows():
            block_id = int(block["block_lgd"])
            block_name = block["block_name"]
            wb = weights[weights["block_lgd"] == block_id]
            cols = wb["column_index_zero_based"].astype(int).tolist()
            w = wb.set_index("column_index_zero_based")[
                "weight_within_intersected_area"
            ].reindex(cols).to_numpy(dtype=float)

            values = climate[cols].to_numpy(dtype=float)
            valid = np.isfinite(values)
            numerator = np.nansum(values * w.reshape(1, -1), axis=1)
            denominator = (valid * w.reshape(1, -1)).sum(axis=1)

            result[f"{block_id}_{block_name}"] = np.divide(
                numerator,
                denominator,
                out=np.full(len(climate), np.nan),
                where=denominator > 0
            )

        # Reindex and fill only absent Feb 29 dates
        result = result.reindex(expected_dates)

        for d in missing_dates:
            before = d - pd.Timedelta(days=1)
            after = d + pd.Timedelta(days=1)

            for col in result.columns:
                a = result.at[before, col]
                b = result.at[after, col]
                if pd.notna(a) and pd.notna(b):
                    result.at[d, col] = (a + b) / 2

        if result.isna().any().any():
            raise ValueError(
                f"Missing block values remain: {int(result.isna().sum().sum())}"
            )

        output = result.copy()
        output.insert(0, "Day", output.index.day)
        output.insert(0, "Month", output.index.month)
        output.insert(0, "Year", output.index.year)
        output.insert(0, "Date", output.index.strftime("%Y-%m-%d"))

        temp_file = output_file.with_suffix(".tmp")
        output.to_csv(temp_file, index=False)
        temp_file.replace(output_file)

        audit_records[str(output_file)] = {
            "model": model,
            "scenario": "ssp245",
            "variable": variable,
            "source_file": str(source),
            "output_file": str(output_file),
            "rows": len(output),
            "start_date": str(expected_dates.min().date()),
            "end_date": str(expected_dates.max().date()),
            "missing_source_dates": len(missing_dates),
            "interpolated_feb29_dates": len(missing_dates),
            "missing_block_day_values": int(result.isna().sum().sum()),
            "status": "SUCCESS"
        }
        pd.DataFrame(audit_records.values()).to_csv(audit_file, index=False)

        print(
            f"  SUCCESS | rows={len(output)} | "
            f"Feb 29 dates inserted={len(missing_dates)}"
        )

    except Exception as exc:
        errors.append({
            "model": model,
            "variable": variable,
            "source_file": str(source),
            "error": repr(exc)
        })
        pd.DataFrame(errors).to_csv(error_file, index=False)
        print("  FAILED:", repr(exc))
        traceback.print_exc()

print("\n========== SUMMARY ==========")
outputs = list(outdir.glob("*_AlwarBlocks.csv"))
valid_outputs = [p for p in outputs if output_is_valid(p)]
print("Output files present:", len(outputs))
print("Outputs passing date validation:", len(valid_outputs))
print("Expected:", 39)

if audit_file.exists():
    audit = pd.read_csv(audit_file)
    print("\nAudit status:")
    print(audit["status"].value_counts().to_string())

if errors:
    print("\nErrors in this run:")
    print(pd.DataFrame(errors).to_string(index=False))


## 5. Quality control and Tmax < Tmin diagnosis

**[orig #196]** Mount Drive and install flopy etc.  
<sub>✅ ran OK</sub>

In [ ]:
# ============================================
# 1. MOUNT GOOGLE DRIVE
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================
# 2. INSTALL REQUIRED PACKAGES
# ============================================
%pip install -q flopy pandas numpy matplotlib scipy openpyxl

# ============================================
# 3. VERIFY INSTALLATIONS
# ============================================
import pandas as pd
import numpy as np
import matplotlib
import scipy
import flopy

print("Google Drive mounted successfully")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("SciPy:", scipy.__version__)
print("FloPy:", flopy.__version__)

**[orig #197]** Library version check  
<sub>✅ ran OK</sub>

In [ ]:
import numpy as np
import pandas as pd
import scipy
import flopy

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("FloPy:", flopy.__version__)

try:
    import numba
    print("Numba:", numba.__version__)
    print("Numba import: SUCCESS")
except Exception as e:
    print("Numba import issue:", repr(e))

# Basic numerical test
a = np.array([1, 2, 3])
print("NumPy calculation:", np.mean(a))

# Basic pandas test
print("Pandas calculation:", pd.Series(a).mean())

print("\nEnvironment diagnostic completed.")

**[orig #198]** List block daily forcing files  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

DAILY_DIR = BASE / "Alwar_grid_analysis/Alwar_block_daily_forcing"

OUTPUT_DIR = BASE / "Alwar_grid_analysis/Climate_Indicators"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = list(DAILY_DIR.glob("*_AlwarBlocks.csv"))

print("Daily files found:", len(files))
print("Output directory:", OUTPUT_DIR)

if len(files) != 39:
    raise ValueError("Expected 39 climate files. Check Google Drive.")

print("\nFirst five files:")
for f in sorted(files)[:5]:
    print(f.name)

**[orig #199]** Columns of the daily files  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DAILY_DIR = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/"
    "Alwar_grid_analysis/Alwar_block_daily_forcing"
)

files = sorted(DAILY_DIR.glob("*_AlwarBlocks.csv"))

blocks = [
    "574_BANSUR", "575_BEHROR", "576_KATHUMAR",
    "577_KISHANGARH BAS", "578_KOTKASIM",
    "579_LAXMANGARH", "580_MANDAWAR",
    "581_NEEMRANA", "582_RAJGARH", "583_RAMGARH",
    "584_RENI", "585_THANAGAZI", "586_TIJARA",
    "587_UMREN"
]

# Use the actual column names from the first file
sample = pd.read_csv(files[0])
print("Columns in daily files:")
print(sample.columns.tolist())

**[orig #200]** Daily QC (missing, duplicates, negative rain, Tmax < Tmin)  
<sub>✅ ran OK</sub>

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. PATHS AND STUDY AREA
# --------------------------------------------------

DAILY_DIR = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/"
    "Alwar_grid_analysis/Alwar_block_daily_forcing"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/"
    "Alwar_grid_analysis/Climate_Indicators"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

blocks = [
    "574_BANSUR", "575_BEHROR", "576_KATHUMAR",
    "577_KISHANGARH BAS", "578_KOTKASIM",
    "579_LAXMANGARH", "580_MANDAWAR",
    "581_NEEMRANA", "582_RAJGARH", "583_RAMGARH",
    "584_RENI", "585_THANAGAZI", "586_TIJARA",
    "587_UMREN"
]

expected_dates = pd.date_range(
    "2015-01-01", "2100-12-31", freq="D"
)

files = sorted(DAILY_DIR.glob("*_AlwarBlocks.csv"))

if len(files) != 39:
    raise ValueError(f"Expected 39 files, found {len(files)}")

# --------------------------------------------------
# 2. READ AND CHECK ALL DAILY FILES
# --------------------------------------------------

data = {}
qc_records = []

for file in files:

    name = file.name
    model = name.split("_ssp245_")[0]

    if "PrecipData" in name:
        variable = "PrecipData"
    elif "TMaxData" in name:
        variable = "TMaxData"
    elif "TMinData" in name:
        variable = "TMinData"
    else:
        raise ValueError(f"Unknown variable: {name}")

    df = pd.read_csv(file)

    required_columns = ["Date", "Year", "Month", "Day"] + blocks

    missing_columns = [
        c for c in required_columns if c not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name}: Missing columns {missing_columns}"
        )

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)

    values = df[blocks].apply(pd.to_numeric, errors="coerce")

    missing_values = int(values.isna().sum().sum())
    infinite_values = int(
        np.isinf(values.to_numpy(dtype=float)).sum()
    )

    date_duplicates = int(df["Date"].duplicated().sum())
    date_mismatch = (
        len(df) != len(expected_dates)
        or not df["Date"].reset_index(drop=True).equals(
            pd.Series(expected_dates, name="Date")
        )
    )

    negative_precip = 0
    extreme_temperature = 0

    if variable == "PrecipData":
        negative_precip = int((values < 0).sum().sum())

    else:
        extreme_temperature = int(
            ((values < -30) | (values > 60)).sum().sum()
        )

    data[(model, variable)] = pd.concat(
        [df[["Date"]], values], axis=1
    )

    qc_records.append({
        "Model": model,
        "Variable": variable,
        "File": name,
        "Rows": len(df),
        "Missing_values": missing_values,
        "Infinite_values": infinite_values,
        "Duplicate_dates": date_duplicates,
        "Date_sequence_error": date_mismatch,
        "Negative_precipitation_values": negative_precip,
        "Extreme_temperature_values": extreme_temperature
    })

# --------------------------------------------------
# 3. CHECK TMAX AND TMIN CONSISTENCY
# --------------------------------------------------

temperature_violations = []

models = sorted(set(model for model, var in data))

for model in models:

    tmax = data[(model, "TMaxData")]
    tmin = data[(model, "TMinData")]

    for block in blocks:

        invalid = (
            tmax[block] < tmin[block]
        )

        count = int(invalid.sum())

        if count > 0:
            temperature_violations.append({
                "Model": model,
                "Block": block,
                "Tmax_below_Tmin_days": count
            })

# --------------------------------------------------
# 4. SAVE QC REPORTS
# --------------------------------------------------

qc_df = pd.DataFrame(qc_records)

qc_path = OUTPUT_DIR / "Daily_Climate_QC_Report.csv"
qc_df.to_csv(qc_path, index=False)

temp_df = pd.DataFrame(temperature_violations)

temp_path = OUTPUT_DIR / "Tmax_Tmin_Consistency_Report.csv"
temp_df.to_csv(temp_path, index=False)

# --------------------------------------------------
# 5. DISPLAY RESULTS
# --------------------------------------------------

print("DAILY CLIMATE QUALITY CONTROL")
print("-----------------------------------")
print("Files checked:", len(qc_df))
print("Expected dates per file:", len(expected_dates))
print("Models checked:", len(models))
print("Blocks checked:", len(blocks))

print("\nQC totals:")
print(qc_df[[
    "Missing_values",
    "Infinite_values",
    "Duplicate_dates",
    "Negative_precipitation_values",
    "Extreme_temperature_values"
]].sum())

print(
    "\nFiles with date sequence errors:",
    int(qc_df["Date_sequence_error"].sum())
)

print(
    "Tmax < Tmin violations:",
    len(temp_df)
)

print("\nQC report saved:")
print(qc_path)

print("\nTemperature consistency report saved:")
print(temp_path)

print("\nFile-level QC:")
display(qc_df)

print("\nTemperature violations:")
display(temp_df.head(20))


**[orig #201]** Daily QC (second version)  
<sub>✅ ran OK</sub>

In [ ]:

import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 1. Define input and output directories
# --------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/"
    "Alwar_grid_analysis"
)

INPUT_DIR = os.path.join(
    BASE_DIR, "Alwar_block_daily_forcing"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "Climate_Indicators"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*_AlwarBlocks.csv"))
)

print("Files detected:", len(files))

if len(files) != 39:
    raise ValueError(
        f"Expected 39 daily files, but found {len(files)}. "
        "Check INPUT_DIR before proceeding."
    )

# --------------------------------------------------
# 2. Define expected dates and block columns
# --------------------------------------------------

expected_dates = pd.date_range(
    start="2015-01-01",
    end="2100-12-31",
    freq="D"
)

blocks = [
    "574_BANSUR",
    "575_BEHROR",
    "576_KATHUMAR",
    "577_KISHANGARH BAS",
    "578_KOTKASIM",
    "579_LAXMANGARH",
    "580_MANDAWAR",
    "581_NEEMRANA",
    "582_RAJGARH",
    "583_RAMGARH",
    "584_RENI",
    "585_THANAGAZI",
    "586_TIJARA",
    "587_UMREN"
]

required_columns = [
    "Date", "Year", "Month", "Day"
] + blocks

# --------------------------------------------------
# 3. Read files and perform daily QC
# --------------------------------------------------

data = {}
qc_results = []

for filepath in files:

    filename = os.path.basename(filepath)

    # Extract model and variable from filename
    variable = next(
        (v for v in ["PrecipData", "TMaxData", "TMinData"]
         if v in filename),
        None
    )

    if variable is None:
        print("Skipping unidentified variable:", filename)
        continue

    model = filename.replace(
        f"_ssp245_{variable}_AlwarBlocks.csv", ""
    )

    df = pd.read_csv(filepath)

    missing_columns = [
        c for c in required_columns if c not in df.columns
    ]

    if missing_columns:
        qc_results.append({
            "Model": model,
            "Variable": variable,
            "File": filename,
            "Rows": len(df),
            "Missing_Columns": ", ".join(missing_columns),
            "Missing_Values": np.nan,
            "Infinite_Values": np.nan,
            "Duplicate_Dates": np.nan,
            "Date_Sequence_Errors": np.nan,
            "Negative_Precipitation": np.nan,
            "Extreme_Temperature_Count": np.nan,
            "Status": "FAIL - Missing columns"
        })
        continue

    # Parse dates
    df["Date"] = pd.to_datetime(
        df["Date"], errors="coerce"
    )

    # Convert climate data to numeric
    for col in blocks:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        )

    # Store the dataframe for later analysis
    df = df.sort_values("Date").reset_index(drop=True)
    data[(model, variable)] = df

    # Count missing and infinite values
    climate_values = df[blocks].to_numpy(dtype=float)

    missing_values = int(
        np.isnan(climate_values).sum()
    )

    infinite_values = int(
        np.isinf(climate_values).sum()
    )

    # Date checks
    duplicate_dates = int(
        df["Date"].duplicated().sum()
    )

    date_sequence_errors = int(
        not np.array_equal(
            df["Date"].to_numpy(),
            expected_dates.to_numpy()
        )
    )

    # Precipitation checks
    negative_precipitation = 0
    if variable == "PrecipData":
        negative_precipitation = int(
            (climate_values < 0).sum()
        )

    # Temperature screening (not automatic deletion)
    extreme_temperature_count = 0
    if variable in ["TMaxData", "TMinData"]:
        extreme_temperature_count = int(
            (
                (climate_values < -30) |
                (climate_values > 60)
            ).sum()
        )

    # Determine file status
    passed = (
        len(df) == len(expected_dates)
        and missing_values == 0
        and infinite_values == 0
        and duplicate_dates == 0
        and date_sequence_errors == 0
        and negative_precipitation == 0
    )

    qc_results.append({
        "Model": model,
        "Variable": variable,
        "File": filename,
        "Rows": len(df),
        "Missing_Columns": "",
        "Missing_Values": missing_values,
        "Infinite_Values": infinite_values,
        "Duplicate_Dates": duplicate_dates,
        "Date_Sequence_Errors": date_sequence_errors,
        "Negative_Precipitation": negative_precipitation,
        "Extreme_Temperature_Count": extreme_temperature_count,
        "Status": "PASS" if passed else "CHECK"
    })

# --------------------------------------------------
# 4. Tmax versus Tmin consistency
# --------------------------------------------------

temperature_results = []

models = sorted(set(
    model for model, variable in data.keys()
))

for model in models:

    if ((model, "TMaxData") not in data or
        (model, "TMinData") not in data):
        continue

    tmax = data[(model, "TMaxData")]
    tmin = data[(model, "TMinData")]

    # Align by date before comparison
    merged = tmax[["Date"] + blocks].merge(
        tmin[["Date"] + blocks],
        on="Date",
        suffixes=("_Tmax", "_Tmin"),
        how="inner"
    )

    violations = 0

    for block in blocks:
        violations += int(
            (
                merged[f"{block}_Tmax"] <
                merged[f"{block}_Tmin"]
            ).sum()
        )

    temperature_results.append({
        "Model": model,
        "Compared_Days": len(merged),
        "Tmax_Less_Than_Tmin_Count": violations,
        "Status": "PASS" if violations == 0 else "CHECK"
    })

# --------------------------------------------------
# 5. Save QC reports
# --------------------------------------------------

qc_df = pd.DataFrame(qc_results)

temperature_df = pd.DataFrame(
    temperature_results
)

qc_path = os.path.join(
    OUTPUT_DIR, "Daily_Climate_QC_Report.csv"
)

temperature_path = os.path.join(
    OUTPUT_DIR, "Tmax_Tmin_Consistency_Report.csv"
)

qc_df.to_csv(qc_path, index=False)
temperature_df.to_csv(temperature_path, index=False)

# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("\n========== DAILY CLIMATE QC ==========")

display(qc_df)

print("\n========== FILE STATUS SUMMARY ==========")
print(qc_df["Status"].value_counts(dropna=False))

print("\n========== TEMPERATURE CONSISTENCY ==========")
display(temperature_df)

print("\n========== OUTPUT FILES ==========")
print(qc_path)
print(temperature_path)


**[orig #203]** Daily QC (final version): 39/39 files pass structure checks; Tmax < Tmin flagged  
<sub>✅ ran OK</sub>

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. PATHS AND EXPECTED DATA
# --------------------------------------------------

DATA_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_block_daily_forcing"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Climate_Indicators"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_DATES = pd.date_range(
    start="2015-01-01",
    end="2100-12-31",
    freq="D"
)

EXPECTED_BLOCKS = [
    "574_BANSUR",
    "575_BEHROR",
    "576_KATHUMAR",
    "577_KISHANGARH BAS",
    "578_KOTKASIM",
    "579_LAXMANGARH",
    "580_MANDAWAR",
    "581_NEEMRANA",
    "582_RAJGARH",
    "583_RAMGARH",
    "584_RENI",
    "585_THANAGAZI",
    "586_TIJARA",
    "587_UMREN"
]

EXPECTED_COLUMNS = [
    "Date", "Year", "Month", "Day"
] + EXPECTED_BLOCKS

files = sorted(glob.glob(
    os.path.join(DATA_DIR, "*_AlwarBlocks.csv")
))

print("Files found:", len(files))
print("Expected files:", 39)

# --------------------------------------------------
# 2. QC FUNCTION
# --------------------------------------------------

qc_results = []
temperature_results = []
data = {}

for file_path in files:

    filename = os.path.basename(file_path)

    # Identify model and variable from filename
    variable = next(
        (v for v in ["PrecipData", "TMaxData", "TMinData"]
         if v in filename),
        None
    )

    model = filename.replace("_ssp245_" + variable + "_AlwarBlocks.csv", "") \
        if variable else "UNKNOWN"

    result = {
        "File": filename,
        "Model": model,
        "Variable": variable,
        "Rows": np.nan,
        "Missing_Values": np.nan,
        "Infinite_Values": np.nan,
        "Duplicate_Dates": np.nan,
        "Invalid_Dates": np.nan,
        "Date_Sequence_Errors": np.nan,
        "Year_Month_Day_Errors": np.nan,
        "Negative_Precipitation": np.nan,
        "Extreme_Temperature_Values": np.nan,
        "Status": "FAIL"
    }

    try:
        df = pd.read_csv(file_path)

        result["Rows"] = len(df)

        # Check columns
        missing_columns = [
            c for c in EXPECTED_COLUMNS if c not in df.columns
        ]

        if missing_columns:
            result["Status"] = "FAIL_MISSING_COLUMNS"
            qc_results.append(result)
            continue

        # Parse dates and numeric values
        df["Date"] = pd.to_datetime(
            df["Date"], errors="coerce"
        )

        numeric_cols = EXPECTED_BLOCKS

        for col in numeric_cols:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            )

        values = df[numeric_cols].to_numpy(dtype=float)

        result["Missing_Values"] = int(
            df[numeric_cols].isna().sum().sum()
        )

        result["Infinite_Values"] = int(
            np.isinf(values).sum()
        )

        result["Invalid_Dates"] = int(
            df["Date"].isna().sum()
        )

        result["Duplicate_Dates"] = int(
            df["Date"].duplicated().sum()
        )

        # Check complete date sequence
        sorted_dates = df["Date"].sort_values().reset_index(drop=True)

        result["Date_Sequence_Errors"] = int(
            len(sorted_dates) != len(EXPECTED_DATES)
            or not sorted_dates.equals(
                pd.Series(EXPECTED_DATES, name=sorted_dates.name)
            )
        )

        # Check Year, Month, Day against Date
        valid_date_rows = df["Date"].notna()

        ymd_errors = (
            (df.loc[valid_date_rows, "Year"].astype("Int64")
             != df.loc[valid_date_rows, "Date"].dt.year)
            |
            (df.loc[valid_date_rows, "Month"].astype("Int64")
             != df.loc[valid_date_rows, "Date"].dt.month)
            |
            (df.loc[valid_date_rows, "Day"].astype("Int64")
             != df.loc[valid_date_rows, "Date"].dt.day)
        )

        result["Year_Month_Day_Errors"] = int(ymd_errors.sum())

        # Variable-specific checks
        if variable == "PrecipData":

            result["Negative_Precipitation"] = int(
                (df[numeric_cols] < 0).sum().sum()
            )

        elif variable in ["TMaxData", "TMinData"]:

            result["Extreme_Temperature_Values"] = int(
                (
                    (df[numeric_cols] < -30)
                    | (df[numeric_cols] > 60)
                ).sum().sum()
            )

            # Retain data for Tmax/Tmin consistency checks
            data[(model, variable)] = df[
                ["Date"] + numeric_cols
            ].copy()

        # Determine file status
        critical_errors = [
            result["Rows"] != len(EXPECTED_DATES),
            result["Missing_Values"] > 0,
            result["Infinite_Values"] > 0,
            result["Duplicate_Dates"] > 0,
            result["Invalid_Dates"] > 0,
            result["Date_Sequence_Errors"] > 0,
            result["Year_Month_Day_Errors"] > 0
        ]

        if variable == "PrecipData":
            critical_errors.append(
                result["Negative_Precipitation"] > 0
            )

        result["Status"] = (
            "PASS" if not any(critical_errors) else "FAIL"
        )

    except Exception as e:
        result["Status"] = "ERROR: " + str(e)

    qc_results.append(result)


# --------------------------------------------------
# 3. CHECK TMAX AGAINST TMIN
# --------------------------------------------------

for model in sorted(set(
    m for m, v in data.keys()
)):

    if (model, "TMaxData") not in data or \
       (model, "TMinData") not in data:
        continue

    tmax = data[(model, "TMaxData")]
    tmin = data[(model, "TMinData")]

    merged = tmax.merge(
        tmin,
        on="Date",
        suffixes=("_Tmax", "_Tmin"),
        validate="one_to_one"
    )

    for block in EXPECTED_BLOCKS:

        violations = (
            merged[f"{block}_Tmax"]
            < merged[f"{block}_Tmin"]
        )

        temperature_results.append({
            "Model": model,
            "Block": block,
            "Tmax_Tmin_Violations": int(violations.sum())
        })


# --------------------------------------------------
# 4. SAVE REPORTS
# --------------------------------------------------

qc_report = pd.DataFrame(qc_results)

temperature_report = pd.DataFrame(
    temperature_results
)

qc_path = os.path.join(
    OUTPUT_DIR,
    "Daily_Climate_QC_Report.csv"
)

temp_path = os.path.join(
    OUTPUT_DIR,
    "Tmax_Tmin_Consistency_Report.csv"
)

qc_report.to_csv(qc_path, index=False)
temperature_report.to_csv(temp_path, index=False)


# --------------------------------------------------
# 5. DISPLAY RESULTS
# --------------------------------------------------

print("\n========== DAILY CLIMATE QC ==========")

print("Files processed:", len(qc_report))
print("PASS:", (qc_report["Status"] == "PASS").sum())
print("FAIL:", (qc_report["Status"] == "FAIL").sum())

display(qc_report)

print("\n========== TMAX / TMIN CHECK ==========")

if not temperature_report.empty:
    print(
        "Total Tmax < Tmin violations:",
        temperature_report["Tmax_Tmin_Violations"].sum()
    )
    display(
        temperature_report[
            temperature_report["Tmax_Tmin_Violations"] > 0
        ]
    )
else:
    print("No complete Tmax/Tmin model pairs were available.")

print("\nReports saved:")
print(qc_path)
print(temp_path)

**[orig #204]** Tmax < Tmin diagnostics per model/block  
<sub>✅ ran OK</sub>

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

DATA_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_block_daily_forcing"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Climate_Indicators"
)

BLOCKS = [
    "574_BANSUR", "575_BEHROR", "576_KATHUMAR",
    "577_KISHANGARH BAS", "578_KOTKASIM",
    "579_LAXMANGARH", "580_MANDAWAR", "581_NEEMRANA",
    "582_RAJGARH", "583_RAMGARH", "584_RENI",
    "585_THANAGAZI", "586_TIJARA", "587_UMREN"
]

files = glob.glob(os.path.join(DATA_DIR, "*_AlwarBlocks.csv"))

violations = []

for model in sorted({
    os.path.basename(f).split("_ssp245_")[0] for f in files
}):

    tmax_file = os.path.join(
        DATA_DIR, f"{model}_ssp245_TMaxData_AlwarBlocks.csv"
    )
    tmin_file = os.path.join(
        DATA_DIR, f"{model}_ssp245_TMinData_AlwarBlocks.csv"
    )

    if not (os.path.exists(tmax_file) and os.path.exists(tmin_file)):
        continue

    tmax = pd.read_csv(tmax_file, parse_dates=["Date"])
    tmin = pd.read_csv(tmin_file, parse_dates=["Date"])

    merged = tmax[["Date"] + BLOCKS].merge(
        tmin[["Date"] + BLOCKS],
        on="Date",
        suffixes=("_Tmax", "_Tmin"),
        validate="one_to_one"
    )

    for block in BLOCKS:
        max_values = merged[f"{block}_Tmax"]
        min_values = merged[f"{block}_Tmin"]

        mask = max_values < min_values

        for idx in merged.index[mask]:
            violations.append({
                "Model": model,
                "Date": merged.loc[idx, "Date"],
                "Block": block,
                "Tmax_C": max_values.loc[idx],
                "Tmin_C": min_values.loc[idx],
                "Reversal_Magnitude_C":
                    min_values.loc[idx] - max_values.loc[idx]
            })

diagnostic = pd.DataFrame(violations)

if diagnostic.empty:
    print("No Tmax < Tmin violations found.")
else:
    diagnostic = diagnostic.sort_values(
        ["Model", "Date", "Block"]
    )

    report_path = os.path.join(
        OUTPUT_DIR,
        "Tmax_Tmin_Violation_Details.csv"
    )
    diagnostic.to_csv(report_path, index=False)

    print("Total violations:", len(diagnostic))
    print("\nReversal magnitude (°C):")
    display(
        diagnostic["Reversal_Magnitude_C"].describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        ).to_frame()
    )

    print("\nViolations by model:")
    display(
        diagnostic.groupby("Model")
        .agg(
            Count=("Date", "size"),
            Mean_Reversal_C=("Reversal_Magnitude_C", "mean"),
            Maximum_Reversal_C=("Reversal_Magnitude_C", "max")
        )
        .sort_values("Count", ascending=False)
    )

    print("\nLargest 30 reversals:")
    display(
        diagnostic.nlargest(30, "Reversal_Magnitude_C")
    )

    print("\nDetailed report saved to:")
    print(report_path)

**[orig #205]** Trace specific Tmax < Tmin dates in source files  
<sub>✅ ran OK</sub>

In [ ]:
import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

SOURCE_DIR = os.path.join(BASE, "India")
OUTPUT_DIR = os.path.join(
    BASE, "Alwar_grid_analysis", "Climate_Indicators"
)

cases = [
    ("BCC-CSM2-MR", "2027-12-28"),
    ("EC-Earth3", "2016-01-07")
]

for model, date in cases:

    print("\n" + "=" * 65)
    print(model, date)

    for variable in ["TMaxData", "TMinData"]:

        path = os.path.join(
            SOURCE_DIR,
            "ssp245",
            variable,
            f"{model}_ssp245_{variable}"
        )

        if not os.path.exists(path):
            print("Source file not found:", path)
            continue

        # Read metadata and daily data separately
        with open(path, "r") as f:
            metadata = [next(f).split(), next(f).split()]

        df = pd.read_csv(
            path,
            sep=r"\s+",
            skiprows=2,
            header=None
        )

        df.columns = (
            ["Year", "Month", "Day"]
            + [f"Grid_{i}" for i in range(df.shape[1] - 3)]
        )

        df["Date"] = pd.to_datetime(
            df[["Year", "Month", "Day"]]
        )

        target = pd.Timestamp(date)

        nearby = df[
            (df["Date"] >= target - pd.Timedelta(days=2))
            & (df["Date"] <= target + pd.Timedelta(days=2))
        ]

        print("\nVariable:", variable)
        print("Metadata row 1 (first 8 entries):", metadata[0][:8])
        print("Metadata row 2 (first 8 entries):", metadata[1][:8])

        print(
            nearby[["Date", "Grid_0", "Grid_1", "Grid_2",
                    "Grid_3", "Grid_4"]].to_string(index=False)
        )

**[orig #207]** Compare the three processing stages for one file  
<sub>✅ ran OK</sub>

In [ ]:
import os
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

folders = [
    "Alwar_daily_extracted",
    "Alwar_daily_leapday_corrected",
    "Alwar_block_daily_forcing"
]

files_to_check = [
    ("Alwar_daily_extracted", "BCC-CSM2-MR_ssp245_TMaxData_Alwar.csv"),
    ("Alwar_daily_leapday_corrected", "BCC-CSM2-MR_ssp245_TMaxData_Alwar.csv"),
    ("Alwar_block_daily_forcing", "BCC-CSM2-MR_ssp245_TMaxData_AlwarBlocks.csv")
]

for folder, filename in files_to_check:
    path = os.path.join(BASE, folder, filename)

    print("\nFOLDER:", folder)

    if os.path.exists(path):
        df = pd.read_csv(path, nrows=3)
        print("Columns:", df.columns.tolist()[:12])
        print(df.head(2).to_string(index=False))
    else:
        print("File not found:", path)

**[orig #208]** Stage-by-stage comparison for affected models  
<sub>✅ ran OK</sub>

In [ ]:
import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

FOLDERS = {
    "Daily_extracted": "Alwar_daily_extracted",
    "Leapday_corrected": "Alwar_daily_leapday_corrected",
    "Block_forcing": "Alwar_block_daily_forcing"
}

CASES = [
    ("BCC-CSM2-MR", "2027-12-28",
     ["577_KISHANGARH BAS", "578_KOTKASIM",
      "580_MANDAWAR", "586_TIJARA"]),

    ("EC-Earth3", "2016-01-07",
     ["578_KOTKASIM", "586_TIJARA"])
]

def read_stage(folder, model, variable):

    if folder == "Alwar_block_daily_forcing":
        filename = (
            f"{model}_ssp245_{variable}_AlwarBlocks.csv"
        )
    else:
        filename = (
            f"{model}_ssp245_{variable}_Alwar.csv"
        )

    path = os.path.join(BASE, folder, filename)

    if not os.path.exists(path):
        print("Missing file:", path)
        return None

    df = pd.read_csv(path)

    if "Date" not in df.columns:
        df["Date"] = pd.to_datetime(
            dict(
                year=df["Year"],
                month=df["Month"],
                day=df["Day"]
            ),
            errors="coerce"
        )
    else:
        df["Date"] = pd.to_datetime(
            df["Date"], errors="coerce"
        )

    return df


records = []

for model, date, blocks in CASES:

    target_date = pd.Timestamp(date)

    for variable in ["TMaxData", "TMinData"]:

        for stage_name, folder in FOLDERS.items():

            df = read_stage(folder, model, variable)

            if df is None:
                continue

            selected = df[df["Date"] == target_date]

            if selected.empty:
                print(
                    "Date not found:",
                    model, variable, stage_name, date
                )
                continue

            for block in blocks:

                if block not in df.columns:
                    continue

                value = pd.to_numeric(
                    selected.iloc[0][block],
                    errors="coerce"
                )

                records.append({
                    "Model": model,
                    "Date": date,
                    "Variable": variable,
                    "Stage": stage_name,
                    "Block": block,
                    "Value_C": value
                })


trace = pd.DataFrame(records)

print("\nSOURCE-TO-FINAL TEMPERATURE COMPARISON")

if not trace.empty:

    comparison = trace.pivot_table(
        index=["Model", "Date", "Block"],
        columns=["Stage", "Variable"],
        values="Value_C",
        aggfunc="first"
    )

    display(comparison.round(4))

    output = os.path.join(
        BASE,
        "Climate_Indicators",
        "Severe_Temperature_Anomaly_Trace.csv"
    )

    os.makedirs(os.path.dirname(output), exist_ok=True)
    trace.to_csv(output, index=False)

    print("\nReport saved:", output)

else:
    print("No comparison records were generated.")

**[orig #209]** Weights table columns; affected block–grid mappings  
<sub>✅ ran OK</sub>

In [ ]:
import os
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

WEIGHT_FILE = os.path.join(
    BASE,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

weights = pd.read_csv(WEIGHT_FILE)

print("Available columns:")
print(weights.columns.tolist())

print("\nAffected block-grid mappings:")

affected_blocks = [
    "577_KISHANGARH BAS",
    "578_KOTKASIM",
    "580_MANDAWAR",
    "586_TIJARA"
]

block_col = next(
    c for c in weights.columns
    if "block" in c.lower() and "name" in c.lower()
)

display(
    weights[
        weights[block_col].isin(affected_blocks)
    ]
)

**[orig #210]** Unique block names / LGD codes in the weights file  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import os

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

WEIGHT_FILE = os.path.join(
    BASE,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

weights = pd.read_csv(WEIGHT_FILE)

print("Unique block names:")
print(weights["block_name"].unique())

print("\nUnique LGD codes:")
print(weights["block_lgd"].unique())

print("\nFirst 15 mapping records:")
display(weights.head(15))

print("\nAffected LGD blocks:")
display(
    weights[
        weights["block_lgd"].isin([577, 578, 580, 586])
    ].sort_values(
        ["block_lgd", "column_index_zero_based"]
    )
)

**[orig #212]** Diagnose grid-column mapping (weights vs extracted columns)  
<sub>✅ ran OK</sub>

In [ ]:
# ================================================================
# DIAGNOSE GRID-COLUMN MAPPING
# Compare spatial-weight indices with extracted CSV columns
# ================================================================

import os
import re
import pandas as pd
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

ANALYSIS_DIR = os.path.join(BASE_DIR, "Alwar_grid_analysis")

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

EXTRACTED_DIR = os.path.join(
    ANALYSIS_DIR, "Alwar_daily_extracted"
)

# ------------------------------------------------
# 1. Read spatial weights
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

weights["block_lgd"] = weights["block_lgd"].astype(int)
weights["column_index_zero_based"] = (
    weights["column_index_zero_based"].astype(int)
)

required_indices = sorted(
    weights["column_index_zero_based"].unique()
)

print("Unique grid indices required:", len(required_indices))
print("Required grid indices:")
print(required_indices)

# ------------------------------------------------
# 2. Inspect the BCC-CSM2-MR Tmax file
# ------------------------------------------------

filename = "BCC-CSM2-MR_ssp245_TMaxData_Alwar.csv"

filepath = os.path.join(EXTRACTED_DIR, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(filepath)

df = pd.read_csv(filepath, nrows=5)

print("\nFile:", filepath)
print("\nFirst 20 column names:")
print(df.columns.tolist()[:20])

# ------------------------------------------------
# 3. Extract grid-column indices from actual headers
# ------------------------------------------------

actual_grid_columns = {}

for col in df.columns:

    match = re.fullmatch(r"grid_col_(\d+)", str(col))

    if match:
        actual_grid_columns[int(match.group(1))] = col

actual_indices = sorted(actual_grid_columns.keys())

print("\nNumber of grid columns found:", len(actual_indices))
print("First 20 actual grid indices:", actual_indices[:20])
print("Last 20 actual grid indices:", actual_indices[-20:])

# ------------------------------------------------
# 4. Identify missing and extra indices
# ------------------------------------------------

missing_indices = sorted(
    set(required_indices) - set(actual_indices)
)

extra_indices = sorted(
    set(actual_indices) - set(required_indices)
)

print("\nGrid indices required but missing:")
print(missing_indices)

print("\nExtra grid indices in extracted file:")
print(extra_indices)

# ------------------------------------------------
# 5. Check possible one-based versus zero-based indexing
# ------------------------------------------------

print("\nIndex alignment diagnostics:")

for shift in [-1, 0, 1]:

    shifted = [i + shift for i in required_indices]

    matched = [
        i for i in shifted
        if i in actual_indices
    ]

    print(
        f"Shift {shift:+d}: "
        f"{len(matched)} of {len(shifted)} indices match"
    )

# ------------------------------------------------
# 6. Inspect mappings for affected blocks
# ------------------------------------------------

affected_lgd = [577, 578, 580, 586]

affected = weights[
    weights["block_lgd"].isin(affected_lgd)
].copy()

print("\nAffected block mappings:")
display(
    affected[
        [
            "block_lgd",
            "block_name",
            "column_index_zero_based",
            "weight_within_intersected_area"
        ]
    ].sort_values(
        ["block_lgd", "column_index_zero_based"]
    )
)

# ------------------------------------------------
# 7. Save diagnostic report
# ------------------------------------------------

diagnostic = pd.DataFrame({
    "Required_Grid_Index": required_indices
})

diagnostic["Expected_Column"] = diagnostic[
    "Required_Grid_Index"
].apply(lambda x: f"grid_col_{x}")

diagnostic["Present_In_Extracted_File"] = (
    diagnostic["Required_Grid_Index"].isin(actual_indices)
)

diagnostic["Column_Minus_One_Present"] = (
    diagnostic["Required_Grid_Index"].apply(
        lambda x: x - 1 in actual_indices
    )
)

diagnostic["Column_Plus_One_Present"] = (
    diagnostic["Required_Grid_Index"].apply(
        lambda x: x + 1 in actual_indices
    )
)

output_file = os.path.join(
    ANALYSIS_DIR,
    "Climate_Indicators",
    "Grid_Column_Mapping_Diagnostic.csv"
)

os.makedirs(os.path.dirname(output_file), exist_ok=True)

diagnostic.to_csv(output_file, index=False)

print("\nDiagnostic report saved:")
print(output_file)

**[orig #214]** Safe audit of all extracted climate files  
<sub>✅ ran OK</sub>

In [ ]:
# ================================================================
# SAFE AUDIT OF ALL EXTRACTED CLIMATE FILES
# Handles empty files, missing headers and incomplete grid columns
# Does not modify or delete any source files
# ================================================================

import os
import re
import glob
import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. DIRECTORIES
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

ANALYSIS_DIR = os.path.join(BASE_DIR, "Alwar_grid_analysis")

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

EXTRACTED_DIR = os.path.join(
    ANALYSIS_DIR, "Alwar_daily_extracted"
)

OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR, "Climate_Indicators"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. READ REQUIRED GRID INDICES
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

required_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

required_columns = {
    f"grid_col_{i}" for i in required_indices
}

print("Required unique grid cells:", len(required_indices))
print("Required indices:", required_indices)

# ------------------------------------------------
# 3. FIND FILES
# ------------------------------------------------

files = sorted(
    glob.glob(os.path.join(EXTRACTED_DIR, "*.csv"))
)

print("\nTotal CSV files found:", len(files))

# ------------------------------------------------
# 4. AUDIT FILES SAFELY
# ------------------------------------------------

audit_records = []
empty_files = []
unreadable_files = []

for filepath in files:

    filename = os.path.basename(filepath)
    file_size = os.path.getsize(filepath)

    # Empty file check
    if file_size == 0:

        empty_files.append(filename)

        audit_records.append({
            "File": filename,
            "File_Size_Bytes": file_size,
            "Present_Cells": 0,
            "Missing_Cells": len(required_columns),
            "Missing_Grid_Columns": "ALL",
            "Status": "EMPTY_FILE"
        })

        continue

    try:

        # Read only CSV header
        header = pd.read_csv(
            filepath,
            nrows=0
        )

        actual_columns = set(header.columns)

        actual_grid_columns = {
            col for col in actual_columns
            if re.fullmatch(r"grid_col_\d+", str(col))
        }

        missing = sorted(
            required_columns - actual_grid_columns
        )

        present = sorted(
            required_columns.intersection(actual_grid_columns)
        )

        status = (
            "PASS"
            if len(missing) == 0
            else "INCOMPLETE"
        )

        audit_records.append({
            "File": filename,
            "File_Size_Bytes": file_size,
            "Present_Cells": len(present),
            "Missing_Cells": len(missing),
            "Missing_Grid_Columns": ", ".join(missing),
            "Status": status
        })

    except pd.errors.EmptyDataError:

        unreadable_files.append(filename)

        audit_records.append({
            "File": filename,
            "File_Size_Bytes": file_size,
            "Present_Cells": 0,
            "Missing_Cells": len(required_columns),
            "Missing_Grid_Columns": "NO READABLE HEADER",
            "Status": "EMPTY_OR_NO_HEADER"
        })

    except Exception as e:

        unreadable_files.append(filename)

        audit_records.append({
            "File": filename,
            "File_Size_Bytes": file_size,
            "Present_Cells": np.nan,
            "Missing_Cells": np.nan,
            "Missing_Grid_Columns": str(e),
            "Status": "READ_ERROR"
        })

audit = pd.DataFrame(audit_records)

# ------------------------------------------------
# 5. DISPLAY SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print(audit["Status"].value_counts(dropna=False))

print("\nEmpty files:")
print(empty_files if empty_files else "None")

print("\nUnreadable files:")
print(unreadable_files if unreadable_files else "None")

print("\nIncomplete or problematic files:")

problem_files = audit[
    audit["Status"] != "PASS"
].copy()

display(problem_files)

# ------------------------------------------------
# 6. CHECK EXPECTED 39 MODEL-VARIABLE FILES
# ------------------------------------------------

models = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

variables = [
    "PrecipData",
    "TMaxData",
    "TMinData"
]

expected_files = [
    f"{model}_ssp245_{variable}_Alwar.csv"
    for model in models
    for variable in variables
]

actual_filenames = {
    os.path.basename(f) for f in files
}

missing_expected = sorted(
    set(expected_files) - actual_filenames
)

unexpected_files = sorted(
    actual_filenames - set(expected_files)
)

print("\n" + "=" * 70)
print("EXPECTED FILE CHECK")
print("=" * 70)

print("Expected files:", len(expected_files))
print("Found files:", len(actual_filenames))

print("\nMissing expected files:")
print(missing_expected if missing_expected else "None")

print("\nUnexpected files:")
print(unexpected_files if unexpected_files else "None")

# ------------------------------------------------
# 7. SAVE AUDIT REPORTS
# ------------------------------------------------

audit_file = os.path.join(
    OUTPUT_DIR,
    "All_Extracted_Files_Grid_Cell_Audit.csv"
)

missing_file_report = os.path.join(
    OUTPUT_DIR,
    "Missing_Expected_Extracted_Files.csv"
)

audit.to_csv(audit_file, index=False)

pd.DataFrame({
    "Missing_Expected_File": missing_expected
}).to_csv(missing_file_report, index=False)

print("\nReports saved:")
print(audit_file)
print(missing_file_report)

print("\nAUDIT COMPLETE")

**[orig #215]** Inspect original source files for the 22 required cells  
<sub>✅ ran OK</sub>

In [ ]:
# ================================================================
# INSPECT ORIGINAL SOURCE FILES FOR ALL 22 REQUIRED GRID CELLS
# Does not modify any existing files
# ================================================================

import os
import glob
import numpy as np
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

ANALYSIS_DIR = os.path.join(BASE_DIR, "Alwar_grid_analysis")

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

SOURCE_DIR = os.path.join(BASE_DIR, "India")

OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR, "Climate_Indicators"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 1. Read required grid indices
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

required_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

print("Required grid indices:", required_indices)
print("Number required:", len(required_indices))

# ------------------------------------------------
# 2. Identify source files
# ------------------------------------------------

source_files = sorted(
    glob.glob(os.path.join(SOURCE_DIR, "**", "*"), recursive=True)
)

source_files = [
    f for f in source_files
    if os.path.isfile(f)
    and os.path.basename(f).endswith(
        ("PrecipData", "TMaxData", "TMinData")
    )
]

print("\nOriginal source files found:", len(source_files))

# ------------------------------------------------
# 3. Inspect source file headers
# ------------------------------------------------

records = []

for filepath in source_files:

    filename = os.path.basename(filepath)

    try:

        with open(filepath, "r") as f:

            first_line = f.readline().strip().split()
            second_line = f.readline().strip().split()

        # Source format: three date columns followed by grid cells
        if len(first_line) != len(second_line):
            raise ValueError(
                "Longitude and latitude metadata lengths differ"
            )

        number_of_grid_cells = len(first_line) - 3

        missing = [
            i for i in required_indices
            if i >= number_of_grid_cells
        ]

        records.append({
            "File": filename,
            "Number_of_Grid_Cells": number_of_grid_cells,
            "Number_of_Required_Cells": len(required_indices),
            "Missing_Grid_Indices": ", ".join(map(str, missing)),
            "Status": (
                "PASS"
                if len(missing) == 0
                else "INCOMPLETE"
            )
        })

    except Exception as e:

        records.append({
            "File": filename,
            "Number_of_Grid_Cells": np.nan,
            "Number_of_Required_Cells": len(required_indices),
            "Missing_Grid_Indices": str(e),
            "Status": "READ_ERROR"
        })

source_audit = pd.DataFrame(records)

print("\nSOURCE FILE AUDIT")
display(source_audit)

print("\nSOURCE AUDIT SUMMARY")
print(source_audit["Status"].value_counts())

# ------------------------------------------------
# 4. Save report
# ------------------------------------------------

report_path = os.path.join(
    OUTPUT_DIR,
    "Original_Source_Grid_Cell_Audit.csv"
)

source_audit.to_csv(report_path, index=False)

print("\nReport saved:")
print(report_path)

## 6. Rebuild from source (unfinished; superseded)

**[orig #217]** Rebuild of the 39 cell files from the original files — stopped after the first file, and it reads the offset columns (`3 + index`); superseded by `03_climate_forcing_corrected`  
<sub>✅ ran OK</sub>

In [ ]:
# ================================================================
# REBUILD ALL 39 ALWAR CMIP6 GRID-CELL DATASETS
# Correct exact model-directory matching
# 13 models x 3 variables x 22 grid cells
# ================================================================

import os
import glob
import numpy as np
import pandas as pd

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

SOURCE_DIR = os.path.join(BASE_DIR, "India")

ANALYSIS_DIR = os.path.join(
    BASE_DIR, "Alwar_grid_analysis"
)

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR,
    "Alwar_daily_extracted_REBUILT"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. MODEL AND VARIABLE LISTS
# ------------------------------------------------

models = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

variables = [
    "PrecipData",
    "TMaxData",
    "TMinData"
]

# ------------------------------------------------
# 3. READ SPATIAL WEIGHTS
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

required_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

if len(required_indices) != 22:
    raise ValueError(
        f"Expected 22 grid cells; found {len(required_indices)}"
    )

print("Required grid cells:", len(required_indices))
print("Grid indices:", required_indices)

# ------------------------------------------------
# 4. FIND SOURCE FILES USING EXACT DIRECTORY MATCHING
# ------------------------------------------------

source_map = {}

for model in models:

    model_directory = os.path.join(
        SOURCE_DIR,
        model,
        "ssp245"
    )

    if not os.path.isdir(model_directory):
        raise FileNotFoundError(
            f"Model directory not found: {model_directory}"
        )

    for variable in variables:

        source_path = os.path.join(
            model_directory,
            variable
        )

        if not os.path.isfile(source_path):
            raise FileNotFoundError(
                f"Missing source file: {source_path}"
            )

        source_map[(model, variable)] = source_path

if len(source_map) != 39:
    raise RuntimeError(
        f"Expected 39 source files; found {len(source_map)}"
    )

print("\nAll 39 source files located successfully.")

# ------------------------------------------------
# 5. EXTRACT REQUIRED GRID CELLS
# ------------------------------------------------

inventory = []

for (model, variable), source_path in source_map.items():

    print(f"\nProcessing {model} | {variable}")

    # Original source:
    # Two metadata rows followed by daily data.
    # Columns 0,1,2 = Year, Month, Day.
    # Columns 3 onward = 4641 grid cells.

    raw = pd.read_csv(
        source_path,
        sep=r"\s+",
        header=None,
        skiprows=2,
        dtype=np.float64
    )

    expected_columns = 3 + 4641

    if raw.shape[1] != expected_columns:
        raise ValueError(
            f"Incorrect source width for {model} {variable}: "
            f"{raw.shape[1]} instead of {expected_columns}"
        )

    # Extract date information
    date_data = raw.iloc[:, :3].copy()

    date_data.columns = [
        "Year",
        "Month",
        "Day"
    ]

    date_data = date_data.astype(int)

    dates = pd.to_datetime(
        dict(
            year=date_data["Year"],
            month=date_data["Month"],
            day=date_data["Day"]
        ),
        errors="raise"
    )

    # Extract all 22 required grid cells
    source_positions = [
        3 + i for i in required_indices
    ]

    selected_data = raw.iloc[
        :,
        source_positions
    ].copy()

    selected_data.columns = [
        f"grid_col_{i}"
        for i in required_indices
    ]

    # Combine dates and grid values
    output = pd.concat(
        [
            date_data.reset_index(drop=True),
            selected_data.reset_index(drop=True)
        ],
        axis=1
    )

    output.insert(0, "Date", dates)

    # Date validation
    if output["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates in {model} {variable}"
        )

    if not output["Date"].is_monotonic_increasing:
        raise ValueError(
            f"Unordered dates in {model} {variable}"
        )

    # Numerical validation
    grid_columns = [
        f"grid_col_{i}"
        for i in required_indices
    ]

    values = output[grid_columns].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            f"Missing or infinite grid values in {model} {variable}"
        )

    # Save rebuilt data
    output_filename = (
        f"{model}_ssp245_{variable}_Alwar_REBUILT.csv"
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        output_filename
    )

    output.to_csv(
        output_path,
        index=False
    )

    inventory.append({
        "Model": model,
        "Variable": variable,
        "Source_File": source_path,
        "Output_File": output_filename,
        "Rows": len(output),
        "Grid_Cells": len(required_indices),
        "Columns": len(output.columns),
        "First_Date": output["Date"].min(),
        "Last_Date": output["Date"].max(),
        "Status": "PASS"
    })

# ------------------------------------------------
# 6. SAVE INVENTORY
# ------------------------------------------------

inventory_df = pd.DataFrame(inventory)

inventory_path = os.path.join(
    OUTPUT_DIR,
    "Rebuilt_Extraction_Inventory.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

# ------------------------------------------------
# 7. FINAL VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("REBUILT EXTRACTION SUMMARY")
print("=" * 70)

display(inventory_df)

print("\nTotal files:", len(inventory_df))
print("Required grid cells:", len(required_indices))
print("Successful files:", (inventory_df["Status"] == "PASS").sum())

print("\nRows per file:")
print(inventory_df["Rows"].value_counts().sort_index())

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nInventory saved:")
print(inventory_path)

if len(inventory_df) != 39:
    raise RuntimeError("Not all 39 files were rebuilt.")

if not (inventory_df["Status"] == "PASS").all():
    raise RuntimeError("One or more files failed validation.")

print("\nSUCCESS: All 39 datasets rebuilt successfully.")
